# BCT Hackathon User Modelling 
> Version : 5
## Goal 
Build an agent that understands users deeply enough to simulate their reviews — capturing tone, rating behaviour, and contextual nuance.
- Simulate star ratings and written reviews for 
unseen items
- Leverage user history, item metadata, and 
contextual signals
- Evaluated on review quality, rating accuracy, 
and behavioural fidelity

### Notebook version 2 
This is the final version of the first build of the it entails downloading the dataset from hugging face (we used the beauty dataset first )
- then perfroming some much needed Exploratory Data Analysis on the data.
- then we created a function to clean the data into review rich data >10 or 100 words reviews
- then we used a hardcoded-review persona builder it builds persona on the reviewer using thier reviews
- then finally it uses gemini 2.5 to generate reviews
- then we then evaluate the data

## Notebook version 3 
- use a multi-agent workflow one agent gets the persona another builds the reviews
- we use pydantic to structure the output of the agents so the agents can give us what we want and not unwanted stuff
- check the evaluation data (hope its not leaking we should be spliting the reviews per users )
- try techniques to save tokens 
- also add something that limits character generation the review generated must be within the average character length of the reviewer (dont write more words than a reviewer will write)

## Notebook version 4 
- added rag to ground agent 2 reviews

## Notebook version 5 (Here now)
- try cross-domain reviewing
- also adding this points
- also need to set up a way to define the products for the ai to understand what it is reviewing 
>Point 1 — Unseen item metadata missing from Agent 2
This one is completely valid and it's the same root cause behind your generation drift problem. Agent 2 currently receives the product title and a truncated description. That's not enough for it to write a product-aware review. It falls back to generic category patterns because it doesn't know enough about the specific item.
What Agent 2 actually needs is a structured item card — not just the title but the category, price tier, key features, brand, and any product-specific attributes. For a skincare product that means ingredients, skin type targeting, and claims. For electronics that means core specs. The item card should be built from your metadata join at data preparation time and passed to Agent 2 as a structured block, not a truncated description string.
This is also where your asin2category.json file from the Kaggle dataset becomes useful. You have a 35 million entry lookup table mapping every ASIN to its category. That's a rich source of item context you haven't used yet.

>Point 2 — Rating before text generation
This is the most technically important point of the three and it's currently broken in your system in a subtle way. You're asking Agent 2 to produce rating and review simultaneously in one JSON object. The LLM generates tokens left to right — if it writes the review text before the rating field, it generates whatever review sounds right and then assigns a rating to match. That's backwards.
The fix is to enforce field ordering in your JSON output so rating always appears before review. JSON objects don't have guaranteed key ordering in the spec but in practice LLMs follow the order they see in the example. Your concrete output example in Agent 2's prompt currently shows reasoning, then rating, then title, then review — which is actually already the right order. The problem is that Gemini doesn't always follow it.
The stronger fix is to split this into two sequential generation steps inside Agent 2. First generate the rating with a brief justification. Then generate the review conditioned on that rating. This forces the autoregressive generation to commit to a number and then write text that justifies it — not the reverse. It directly addresses the incoherence problem where positive text gets a 2-star rating.
This also improves your RMSE because the rating decision is now isolated and deliberate rather than being a byproduct of whatever text happened to get generated.

>Point 3 — Contextual signals injection
This one needs to be interpreted carefully against your actual brief. The hackathon brief says "contextual signals" but in the Amazon reviews context this means signals available in the review data itself — things like whether it was a verified purchase, the time gap between purchase and review, whether the user reviewed multiple items from the same brand recently, and their rating trajectory at the time of the review.
It does not mean real-time signals like time of day or device — those aren't in your dataset and you can't simulate them without inventing data, which would hurt your evaluation scores.
What you can inject from your existing data is genuinely useful contextual grounding. How long after their last review is this review being written — are they in an active reviewing period or returning after a long gap? Have they recently reviewed other products in the same category — do they have fresh category context? What is their rating trajectory at this point in time — are they in a generous phase or a critical phase based on their recent reviews? These are all derivable from your cleaned dataset and they add real signal to Agent 2's generation.

>What needs to change in your system
These three points map to three specific changes in priority order.
First, build a proper item card constructor that pulls from your metadata and asin2category lookup. This fixes the product awareness problem and should move your ROUGE-2 score meaningfully.
Second, restructure Agent 2 into two sequential calls — rating generation then review generation. This fixes the coherence problem and should improve RMSE consistency.
Third, add a contextual signals block to Agent 2's prompt that injects the user's reviewing context at the time of the holdout review — their recent rating trend, time since last review, and category familiarity score. This is derivable from your existing data with no additional API calls.
All three are buildable before your deadline and all three have direct mappings to scoring criteria — item metadata improves ROUGE, two-step generation improves RMSE, contextual signals improve behavioural fidelity. Each one earns you points in a different evaluation dimension.
Want to build all three now?

## importing dependencies

In [1]:
# ── INSTALL FIRST ────────────────────────────────────────────────
!pip install datasets
!pip install bert-score
!pip install rouge-score
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.9 MB/s eta 0:00:00


In [2]:
import json 
import re
import hashlib
import os
import faiss
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import time
import random
from google import genai
from google.genai import types
from typing import Optional
from enum import Enum
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field, field_validator


from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

In [3]:
# use the kaggle secret enviroment to secure my api key 

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
HF_Key = UserSecretsClient().get_secret("HF_Key")

In [4]:
def load_category(
    reviews_path  : str,
    meta_path     : str,
    category_name : str,
    nrows         : int = None    # None = load everything, int = load that many rows
) -> pd.DataFrame:
    """
    Loads and joins reviews + metadata for one Amazon category.
    Returns a clean joined dataframe with standardised column names.

    Parameters:
        reviews_path  : path to reviews .jsonl.gz file
        meta_path     : path to metadata .jsonl.gz file
        category_name : label for this category (stored in source_category)
        nrows         : number of review rows to load — None loads everything
    """
    print(f"Loading {category_name}...")
    if nrows:
        print(f"  Row limit : {nrows:,}")
    else:
        print(f"  Row limit : none — loading full dataset")

    # ── LOAD REVIEWS ──────────────────────────────────────────────
    read_kwargs = {
        'lines'       : True,
        'compression' : 'gzip'
    }
    if nrows is not None:
        read_kwargs['nrows'] = nrows

    reviews_df = pd.read_json(reviews_path, **read_kwargs)
    meta_df    = pd.read_json(meta_path, lines=True, compression='gzip')

    print(f"  Reviews loaded  : {len(reviews_df):,}")
    print(f"  Products loaded : {len(meta_df):,}")

    # ── FLATTEN METADATA LIST FIELDS ─────────────────────────────
    meta_df['description_text'] = meta_df['description'].apply(
        lambda x: ' '.join(x) if isinstance(x, list) and x else ''
    )
    meta_df['features_text'] = meta_df['features'].apply(
        lambda x: ' | '.join(x[:3]) if isinstance(x, list) and x else ''
    )

    # ── SLIM METADATA ─────────────────────────────────────────────
    meta_slim = meta_df[[
        'parent_asin',
        'title',
        'description_text',
        'features_text',
        'price',
        'store',
        'main_category'
    ]].drop_duplicates(subset='parent_asin')

    # ── JOIN ──────────────────────────────────────────────────────
    df = reviews_df.merge(meta_slim, on='parent_asin', how='left')

    # ── STANDARDISE COLUMN NAMES ──────────────────────────────────
    df = df.rename(columns={
        'verified': 'verified_purchase',
        'title_x'          : 'review_title',
        'title_y'          : 'product_title',
    })

    # ── TAG SOURCE ────────────────────────────────────────────────
    df['source_category'] = category_name
    # --Drop Image column---------------
    df = df.drop(columns = "images", axis = 1)
    print(f"  Joined rows     : {len(df):,}")
    return df


In [5]:
def normalise_video_games_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups fragmented Video Games metadata categories
    into clean top-level buckets for RAG matching.
    Handles null/NaN values gracefully.
    """

    gaming_cats = {
        'video games', 'software', 'toys & games',
        'buy a kindle', 'audible audiobooks'
    }
    electronics_cats = {
        'computers', 'all electronics', 'cell phones & accessories',
        'home audio & theater', 'amazon devices', 'camera & photo',
        'digital music', 'musical instruments', 'portable audio & accessories',
        'car electronics', 'gps & navigation', 'amazon home'
    }
    beauty_cats = {
        'all beauty', 'premium beauty', 'health & personal care',
        'amazon fashion'
    }
    other_cats = {
        'books', 'movies & tv', 'grocery', 'office products',
        'tools & home improvement', 'sports & outdoors',
        'industrial & scientific', 'pet supplies', 'baby',
        'automotive', 'arts, crafts & sewing', 'appliances',
        'collectible coins'
    }

    def map_category(cat):
        # ── HANDLE NULL / NaN / EMPTY ─────────────────────────────
        if cat is None:
            return 'Other'
        if isinstance(cat, float):        # NaN comes in as float
            return 'Other'
        
        cat_lower = str(cat).strip().lower()
        
        if not cat_lower or cat_lower == 'nan':
            return 'Other'
        if cat_lower in gaming_cats      : return 'Video Games and Software'
        if cat_lower in electronics_cats : return 'Electronics'
        if cat_lower in beauty_cats      : return 'Beauty'
        if cat_lower in other_cats       : return 'Other'
        return 'Other'

    # ── APPLY MAPPING ─────────────────────────────────────────────
    df = df.copy()
    df['main_category'] = df['main_category'].apply(map_category)

    # ── VERIFY NO NULLS REMAIN ────────────────────────────────────
    null_count = df['main_category'].isna().sum()
    if null_count > 0:
        print(f"  ⚠️  {null_count} nulls still present — filling with 'Other'")
        df['main_category'] = df['main_category'].fillna('Other')

    # ── FINAL DISTRIBUTION ────────────────────────────────────────
    print("\nCategory distribution after normalisation:")
    dist = df['main_category'].value_counts()
    for cat, count in dist.items():
        print(f"  {str(cat):<35} {count:,}")

    # ── CONFIRM NO NULLS ──────────────────────────────────────────
    remaining_nulls = df['main_category'].isna().sum()
    print(f"\n  Null values remaining : {remaining_nulls}")

    return df

In [6]:
def merge_categories(
    dataframes       : list[pd.DataFrame],
    drop_duplicates  : bool = True,
    min_text_words   : int  = 10,
    verbose          : bool = True
) -> pd.DataFrame:
    """
    Merges multiple category dataframes into one clean combined dataset.

    Handles:
    - Duplicate reviews (same user reviewing same product across datasets)
    - Column alignment across categories
    - Text quality filter
    - Summary report

    Parameters:
        dataframes      : list of loaded category dataframes
        drop_duplicates : remove same user + same product duplicates
        min_text_words  : minimum words for a review to be kept
        verbose         : print merge summary

    Returns:
        merged_df : single clean combined dataframe
    """

    print("\n" + "=" * 60)
    print("MERGING CATEGORIES")
    print("=" * 60)

    # ── ALIGN COLUMNS ACROSS DATAFRAMES ──────────────────────────
    # get union of all columns
    all_cols = set()
    for df in dataframes:
        all_cols.update(df.columns.tolist())

    # add missing columns as NaN so concat doesn't fail
    aligned = []
    for df in dataframes:
        missing = all_cols - set(df.columns)
        for col in missing:
            df[col] = None
        aligned.append(df)

    # ── CONCAT ───────────────────────────────────────────────────
    merged_df = pd.concat(aligned, ignore_index=True)

    if verbose:
        print(f"\nRaw combined rows : {len(merged_df):,}")
        print(f"Sources           :")
        for cat, count in merged_df['source_category'].value_counts().items():
            print(f"  {cat:<35} {count:,} reviews")

    # ── DROP EXACT DUPLICATES ─────────────────────────────────────
    before = len(merged_df)
    merged_df = merged_df.drop_duplicates()
    if verbose:
        print(f"\nExact duplicates removed  : {before - len(merged_df):,}")

    # ── DROP USER + PRODUCT DUPLICATES ───────────────────────────
    # same user reviewing same product in both datasets
    if drop_duplicates and 'user_id' in merged_df.columns and 'parent_asin' in merged_df.columns:
        before = len(merged_df)
        merged_df = merged_df.drop_duplicates(
            subset  = ['user_id', 'parent_asin'],
            keep    = 'first'
        )
        if verbose:
            print(f"User+product dupes removed: {before - len(merged_df):,}")

    # ── TEXT QUALITY FILTER ───────────────────────────────────────
    if 'text' in merged_df.columns:
        merged_df['text'] = merged_df['text'].fillna('').astype(str)
        before = len(merged_df)
        merged_df = merged_df[
            merged_df['text'].str.split().str.len() >= min_text_words
        ]
        if verbose:
            print(f"Short reviews removed     : {before - len(merged_df):,}")

    # ── RESET INDEX ───────────────────────────────────────────────
    merged_df = merged_df.reset_index(drop=True)

    # ── FINAL SUMMARY ─────────────────────────────────────────────
    if verbose:
        print(f"\n{'=' * 60}")
        print(f"MERGED DATASET READY")
        print(f"{'=' * 60}")
        print(f"  Total rows        : {len(merged_df):,}")
        print(f"  Unique users      : {merged_df['user_id'].nunique():,}")
        print(f"  Unique products   : {merged_df['parent_asin'].nunique():,}")
        print(f"  Categories        : {merged_df['source_category'].nunique()}")

        user_counts = merged_df.groupby('user_id').size()
        viable      = user_counts[user_counts >= 20]
        print(f"  Users with 20+    : {len(viable):,}")
        print(f"  Avg reviews/user  : {user_counts.mean():.1f}")
        print(f"{'=' * 60}\n")

    return merged_df

In [7]:
# ── DOWNLOAD "All Beauty category dataset" ─────────────────────────────────────────────────────
!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz" \
    -O "/kaggle/working/All_Beauty_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz" \
    -O "/kaggle/working/meta_All_Beauty.jsonl.gz"

# confirm sizes — should be several MB each
!ls -lh /kaggle/working/*.gz

# ── DOWNLOAD video_games CATEGORY ──────────────────────────────────────
# pick whichever category suits your project

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz" \
    -O "/kaggle/working/game_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz" \
    -O "/kaggle/working/meta_game.jsonl.gz"


# ── LOAD BOTH CATEGORIES ──────────────────────────────────────────

beauty_df = load_category(
    reviews_path  = '/kaggle/working/All_Beauty_reviews.jsonl.gz',
    meta_path     = '/kaggle/working/meta_All_Beauty.jsonl.gz',
    category_name = 'All Beauty',
    nrows         = None         )

# load first 500k rows — for large categories like Books
game_df = load_category(
    reviews_path  = '/kaggle/working/game_reviews.jsonl.gz',
    meta_path     = '/kaggle/working/meta_game.jsonl.gz',
    category_name = 'Books',
    nrows         = 600_000 
    )


# normalise categories
game_df = normalise_video_games_categories(game_df)


# ── MERGE ─────────────────────────────────────────────────────────

df = merge_categories(
dataframes      = [beauty_df, game_df],
drop_duplicates = True,
min_text_words  = 10,
verbose         = True)

/kaggle/working/All 100%[===================>]  90.07M  36.0MB/s    in 2.5s    
/kaggle/working/met 100%[===================>]  38.02M  30.5MB/s    in 1.2s    
-rw-r--r-- 1 root root 91M Jan 16  2025 /kaggle/working/All_Beauty_reviews.jsonl.gz
-rw-r--r-- 1 root root 39M Jan 16  2025 /kaggle/working/meta_All_Beauty.jsonl.gz
/kaggle/working/gam 100%[===================>] 776.49M  19.5MB/s    in 86s     
/kaggle/working/met 100%[===================>]  98.32M  18.5MB/s    in 5.3s    
Loading All Beauty...
  Row limit : none — loading full dataset
  Reviews loaded  : 701,528
  Products loaded : 112,590
  Joined rows     : 701,528
Loading Books...
  Row limit : 600,000
  Reviews loaded  : 600,000
  Products loaded : 137,269
  Joined rows     : 600,000

Category distribution after normalisation:
  Video Games and Software            395,049
  Electronics                         188,211
  Other                               16,359
  Beauty                              381

  Null values remain

In [8]:
# ── HEALTH CHECK ─────────────────────────────────────────────────
user_counts = df.groupby('user_id').size()
viable      = user_counts[user_counts >= 20]

print("=" * 50)
print("HEALTH CHECK")
print("=" * 50)
print(f"Total reviews         : {len(df):,}")
print(f"Unique users          : {df['user_id'].nunique():,}")
print(f"Unique products       : {df['asin'].nunique():,}")
print(f"Users with 20+ reviews: {len(viable):,}")
print(f"Avg review length     : {df['text'].str.split().str.len().mean():.0f} words")
print(f"Rating distribution   : {df['rating'].value_counts().sort_index().to_dict()}")
#print(f"Missing product title : {df['product_title'].isna().sum():,}")
#print(f"Missing description   : {df['description_text'].isna().sum():,}")
print(f"Verified purchase %   : {df['verified_purchase'].mean()*100:.1f}%")

HEALTH CHECK
Total reviews         : 923,648
Unique users          : 642,216
Unique products       : 169,696
Users with 20+ reviews: 1,315
Avg review length     : 61 words
Rating distribution   : {1: 122970, 2: 60587, 3: 85385, 4: 129772, 5: 524934}
Verified purchase %   : 86.0%


In [9]:
df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,source_category
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,Herbivore - Natural Sea Mist Texturizing Salt ...,"If given the choice, weÕd leave most telltale ...",,NaN,HERBIVORE,All Beauty,All Beauty
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,All Natural Vegan Dry Shampoo Powder - Eco Fri...,,,NaN,Two Goats Apothecary,All Beauty,All Beauty
2,4,Pretty Color,The polish was quiet thick and did not apply s...,B00R8DXL44,B00R8DXL44,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,2020-08-27 22:30:08.138,0,True,"China Glaze Nail Polish, Wanderlust 1381","China Glaze Nail Polish, Wanderlust, 1381, .50...",Light lavender pink nail color with golden shi...,7.1,China Glaze,All Beauty,All Beauty
3,5,Handy,Great for many tasks. I purchased these for m...,B099DRHW5V,B099DRHW5V,AHREXOGQPZDA6354MHH4ETSF3MCQ,2021-09-17 13:31:59.443,0,True,"Disposable Facial Cotton Tissue, 100PCS Cotton...",,,NaN,AYQNMHR,All Beauty,All Beauty
4,3,Meh,These were lightweight and soft but much too s...,B088SZDGXG,B08BBQ29N5,AEYORY2AVPMCPDV57CE337YU5LXA,2021-10-15 05:20:59.292,0,True,Niseyo new Faux Locs 24 Inch Crochet Hair 6 Pa...,,,NaN,Niseyo,All Beauty,All Beauty


## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [10]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (923648, 16)

Column dtypes:
rating                        int64
review_title                 object
text                         object
asin                         object
parent_asin                  object
user_id                      object
timestamp            datetime64[ns]
helpful_vote                  int64
verified_purchase              bool
product_title                object
description_text             object
features_text                object
price                        object
store                        object
main_category                object
source_category              object
dtype: object

Missing values:
rating                    0
review_title              0
text                      0
asin                      0
parent_asin               0
user_id                   0
timestamp                 0
helpful_vote              0
verified_purchase         0
product_title             0
description_text          0
features_text           

### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [11]:
## understanding the schema of the review dataset 
df.iloc[500].to_dict()

{'rating': 1,
 'review_title': 'Too oily for me.',
 'text': "My daughter loves these but it's too greasy for me.",
 'asin': 'B00E2S00ME',
 'parent_asin': 'B00E2S00ME',
 'user_id': 'AEYE3LSUHQEX6BWG224MKVXX6XZQ',
 'timestamp': Timestamp('2020-06-22 16:55:51.873000'),
 'helpful_vote': 0,
 'verified_purchase': True,
 'product_title': 'Josie Maran Argan Creamy Concealer Crayon (Fair 1)',
 'description_text': '',
 'features_text': '',
 'price': 34.97,
 'store': 'Josie Maran',
 'main_category': 'All Beauty',
 'source_category': 'All Beauty'}

In [12]:
#checking the columns in the data set 
df.columns.tolist()

['rating',
 'review_title',
 'text',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'product_title',
 'description_text',
 'features_text',
 'price',
 'store',
 'main_category',
 'source_category']

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona

- the results here are quite worse to build the prototype i will use users with 5 or more reviews 

In [13]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

count    642216.000000
mean          1.438220
std           2.352188
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         544.000000
dtype: float64

In [14]:
# How many viable user?
five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")


The number of users with five reviews or more are 18041,
        The number of users with ten reviews or more are 4877,
        while the number of users with twenty reviews or more are 1315


In [15]:
# Distribution shape
user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

1     537288
2      58690
3      19052
4       9145
5       5119
6       3343
7       2137
8       1513
9       1052
10       835
11       651
12       441
13       394
14       314
15       250
16       210
17       192
18       138
19       137
20       136
Name: count, dtype: int64

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [16]:
# Global rating distribution
df['rating'].value_counts().sort_index()

rating
1    122970
2     60587
3     85385
4    129772
5    524934
Name: count, dtype: int64

In [17]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

count    642216.000000
mean          3.885016
std           1.463379
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [18]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

count    104928.000000
mean          0.845995
std           0.856087
min           0.000000
25%           0.000000
50%           0.707107
75%           1.414214
max           2.828427
Name: rating, dtype: float64

In [19]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

count    1.804100e+04
mean    -7.896293e-03
std      2.629346e-01
min     -1.200000e+00
25%     -1.000000e-01
50%     -1.321078e-16
75%      1.000000e-01
max      1.200000e+00
Name: rating_slope, dtype: float64
trend
insufficient_data        624175
consistent                 6586
increasingly_critical      5815
increasingly_generous      5640
Name: count, dtype: int64


In [20]:
df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,source_category,review_seq
733368,4,Nice addition to WoW,"I have been playing for a long time, but we to...",B000VJTJNE,B00400OFG6,AE22236AFRRSMQIKGG7TPTB75QEA,2009-10-24 14:09:15.000,0,True,World of Warcraft: Wrath of the Lich King Expa...,,Master the necromantic powers of the Death Kni...,18.52,Blizzard Entertainment,Video Games and Software,Books,1
733367,5,Takes Wii Fit to the next level,This game made Wii Fit so much better. Its a ...,B002BS47JE,B002BS47JE,AE22236AFRRSMQIKGG7TPTB75QEA,2010-05-30 19:22:49.000,0,False,Wii Fit Plus,Product Description NOW YOU CAN BUILD YOUR OWN...,Users can input the amount of time they want t...,17.5,Nintendo,Video Games and Software,Books,2
628233,5,family fun,This is a fun game for the whole family especi...,B004LNJ1FC,B004LNJ1FC,AE222H5U2B6JXKSCG4ILOQCHFRBA,2013-06-22 18:39:35.000,0,True,Wii Party with Remote,Wii Party With WIi remote,,None,Nintendo,Video Games and Software,Books,1
922007,5,Excelente juego.,"Excelente juego, nuevamente una entrega sin pr...",B07Y92FSWL,B07SNN8GV5,AE222HFZDH6BPTYFOUWGGU63YSIQ,2019-12-15 14:32:20.876,0,True,Call of Duty: Modern Warfare Battle Pass Editi...,"Prepare to go dark, Modern Warfare is back! Th...",In the visceral and dramatic single-player sto...,None,ACTIVISION,Video Games and Software,Books,1
922006,5,La recomiendo 100%,"La consola es muy funcional, excelente product...",B07XQXZXJC,B01GY35T4S,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-01-04 18:08:57.203,1,True,Xbox One S 1TB Console - NBA 2K20 Bundle - [DI...,Own the Xbox One S NBA 2K20 Bundle and experie...,The previous generation bundle includes: 1TB X...,275.0,Xbox,Electronics,Books,2


In [21]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

"                     timestamp  rating  review_seq                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 text\n850757 2020-03-28 21:08:05.123       5           1                                                            

In [22]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n📊 QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f} ⭐  (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [23]:

"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

USER PROFILE: AHBWH2LBU3NFLD46GKJKIBAHKXEQ

📊 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.05 ⭐  (std: 1.00)
   Avg review length: 104 words
   Rating breakdown: {1: 1, 2: 3, 3: 3, 4: 18, 5: 14}
   Rating trend    : ↑ more generous over time  (early avg: 3.89 → late avg: 4.20)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Aug 2020  |  ⭐⭐⭐⭐ (4/5)  |  237 words
     I try to avoid using traditional files and emery boards
     on my nails. I take a lot of medications that make my
     nails weak and brittle, and any surface that's too
     abrasive wreaks havoc on my fingertips. I can't carry a
     full size glass nail file with me everywhere, so I was
     on the hunt for a travel sized file that could fit in
     my pocket, wallet, or purse. When these came up, I
     thought I'd give them a shot. The pros are the files
     are a great size, they're packaged well, and I see them
     lasting for awhile. 

In [24]:
# random user with 30+ reviews
user_history = profile_user(df)

Randomly selected user: AH4IRPVSP562B476GMLP5GVN4E3A

USER PROFILE: AH4IRPVSP562B476GMLP5GVN4E3A

📊 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.49 ⭐  (std: 0.85)
   Avg review length: 228 words
   Rating breakdown: {2: 2, 3: 3, 4: 8, 5: 26}
   Rating trend    : → consistent rater  (early avg: 4.58 → late avg: 4.40)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Dec 2007  |  ⭐⭐⭐⭐ (4/5)  |  227 words
     We got the Wii for Christmas (the ever-coveted Wii!
     YES!) and immediately struggled with what game to buy
     for our 7 and 3 1/2 year olds...we wanted something we
     could all play together, and something that wouldn't be
     too challenging. After a lot of debating and searching,
     Wii Play was chosen (mainly because it came with
     another wiimote).<br /><br />This is the perfect game
     for people just learning how to use the Wii. The whole
     purpose of the game is for you to learn 

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

In [25]:
# Review length distribution
df['text'].str.split().str.len().describe()

count    923648.000000
mean         60.659999
std         100.076605
min          10.000000
25%          19.000000
50%          32.000000
75%          63.000000
max        5240.000000
Name: text, dtype: float64

In [26]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

0

In [27]:
# Avg text length per user (computationaly expensive to load)
#avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [28]:
#display(avg_text_per_user)

In [29]:
#print(avg_text_per_user.value_counts())


In [30]:
# Verified purchase flag

df['verified_purchase'].value_counts()
# prefer verified = True

verified_purchase
True     794280
False    129368
Name: count, dtype: int64

## Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data 

In [31]:

def clean_amazon_reviews(df, 
                          min_reviews=10, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[ user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining      user_counts         : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [32]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

AMAZON REVIEWS — CLEANING PIPELINE

▶ Starting shape: 923,648 rows × 17 columns

[1] Duplicate rows removed   : 0
    Remaining                : 923,648

[2] Duplicate columns removed: 0
    Remaining columns        : ['rating', 'review_title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description_text', 'features_text', 'price', 'store', 'main_category', 'source_category', 'review_seq']

[3] Rows dropped (missing critical fields): 0
    Remaining                              : 923,648

[4] Rows dropped (empty / short reviews) : 422,260
    Min word count threshold             : 30 words
    Remaining                            : 501,388

[5] Rows dropped (unverified purchases)  : 102,055
    Remaining                            : 399,333

[6] Rows dropped (invalid ratings)       : 0
    Remaining                            : 399,333

[7] Users dropped (< 10 reviews)          : 297,231
    Viable users remaining      u

In [33]:
rich_df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,source_category,review_seq,word_count
0,4,great workout,The game provides one heck of a workout. My fi...,B002BUSVJO,B002JTX8BS,AE25GU3LWQGZJN4NNT5GWAGBN2KA,2010-05-03 16:12:37,0,True,Biggest Loser - Nintendo Wii,"Product Description Utilizes the Wii Remote, N...","Extensive workout customization, a daily calen...",13.0,THQ,Video Games and Software,Books,1,50
1,5,Great alternative to going wireless,I have had several wireless PS2 controllers ov...,B00005MDZ1,B00005MDZ1,AE25GU3LWQGZJN4NNT5GWAGBN2KA,2011-02-06 03:30:35,2,True,PS2 Controller Extension Cable,This cable easily attaches to any PS2 compatib...,Extend your game | Adds 6 feet to the controll...,None,Sony,Video Games and Software,Books,3,47
2,3,"Fun and creative game, too short",I would love to see more games like this. Each...,B00213JM9O,B003YH9V0Q,AE25GU3LWQGZJN4NNT5GWAGBN2KA,2011-03-02 22:19:48,0,True,World of Goo [Online Game Code],Amazon.com You Can’t Stop Progress… World of G...,"Build structures, bridges, cannonballs, zeppel...",None,2D Boy,Video Games and Software,Books,4,121
3,3,so-so,I use the GTA games and Bully as the benchmark...,B0009Z3HYW,B001ELJE6K,AE25GU3LWQGZJN4NNT5GWAGBN2KA,2011-07-11 17:35:52,1,True,GUN - PC,From the Manufacturer Gun is a free-roaming ac...,For 1 player | Free-roaming action-adventure s...,None,ACTIVISION,Video Games and Software,Books,5,123
4,2,Got bored with it quickly,I see this game as a downgrade from Sims III. ...,B004S82O2C,B004S82O2C,AE25GU3LWQGZJN4NNT5GWAGBN2KA,2011-07-15 17:54:32,0,True,The Sims Medieval [Download],From the Manufacturer The Sims Medieval is a b...,,None,Electronic Arts,Video Games and Software,Books,6,35


In [34]:
rich_df.to_csv('rich_users.csv', index=False)

## Split the cleaned data 
 this function splits the cleaned data to the training data(persona_df), the validation data (val_df) and the test data 

In [35]:
import pandas as pd
import numpy as np

def split_user_reviews(clean_df, min_reviews=20, n_test=1, n_val=1):
    """
    Splits cleaned Amazon reviews into three sets per user:
    - Persona set   : bulk history for Agent 1 profiling
    - Validation set: second-to-last reviews for prompt tuning
    - Test set      : last reviews for final evaluation only

    Parameters:
        clean_df    : your cleaned dataframe from clean_amazon_reviews()
        min_reviews : minimum reviews a user needs to be included
        n_test      : number of reviews to hold out for test (default 1)
        n_val       : number of reviews to hold out for validation (default 1)

    Returns:
        persona_df  : DataFrame — bulk user history
        val_df      : DataFrame — validation reviews
        test_df     : DataFrame — test reviews (lock this away)
        split_stats : dict — summary of the split
    """

    print("=" * 60)
    print("TEMPORAL REVIEW SPLIT")
    print("=" * 60)
    print(f"  Min reviews required : {min_reviews}")
    print(f"  Test holdout         : last {n_test} review(s) per user")
    print(f"  Validation holdout   : {n_val} review(s) before test")
    print(f"  Persona minimum      : {min_reviews - n_test - n_val} reviews\n")

    # ── VALIDATE INPUTS ──────────────────────────────────────────
    required_cols = ['user_id', 'timestamp', 'text', 'rating']
    missing       = [c for c in required_cols if c not in clean_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    min_needed = min_reviews + n_test + n_val
    
    # ── SORT BY USER + TIME ──────────────────────────────────────
    df = clean_df.sort_values(
        ['user_id', 'timestamp']
    ).reset_index(drop=True)

    # ── SPLIT PER USER ───────────────────────────────────────────
    persona_rows  = []
    val_rows      = []
    test_rows     = []
    skipped_users = []

    user_groups = df.groupby('user_id')

    for user_id, group in user_groups:
        group = group.reset_index(drop=True)
        n     = len(group)

        # skip users without enough reviews
        if n < min_needed:
            skipped_users.append({
                'user_id'      : user_id,
                'review_count' : n,
                'reason'       : f"needs {min_needed}, has {n}"
            })
            continue

        # temporal split — oldest to newest
        test_slice    = group.iloc[-n_test:]                      # last n_test
        val_slice     = group.iloc[-(n_test + n_val):-n_test]     # before test
        persona_slice = group.iloc[:-(n_test + n_val)]            # everything before

        # tag each split
        test_slice    = test_slice.copy()
        val_slice     = val_slice.copy()
        persona_slice = persona_slice.copy()

        test_slice['split']    = 'test'
        val_slice['split']     = 'validation'
        persona_slice['split'] = 'persona'

        test_rows.append(test_slice)
        val_rows.append(val_slice)
        persona_rows.append(persona_slice)

    # ── BUILD DATAFRAMES ─────────────────────────────────────────
    persona_df = pd.concat(persona_rows,  ignore_index=True) if persona_rows else pd.DataFrame()
    val_df     = pd.concat(val_rows,      ignore_index=True) if val_rows     else pd.DataFrame()
    test_df    = pd.concat(test_rows,     ignore_index=True) if test_rows    else pd.DataFrame()

    # ── SANITY CHECKS ────────────────────────────────────────────
    # 1 — no user should appear in both val and test with the same review
    persona_ids = set(persona_df['user_id'].unique()) if len(persona_df) else set()
    val_ids     = set(val_df['user_id'].unique())     if len(val_df)     else set()
    test_ids    = set(test_df['user_id'].unique())    if len(test_df)    else set()

    # every test user must have a persona
    missing_persona = test_ids - persona_ids
    if missing_persona:
        print(f"⚠️  WARNING: {len(missing_persona)} test users have no persona history")

    # check for timestamp ordering — test must always be after persona
    ordering_violations = 0
    for user_id in list(test_ids)[:50]:    # sample check on first 50 users
        p_times = persona_df[persona_df['user_id'] == user_id]['timestamp']
        t_times = test_df[test_df['user_id'] == user_id]['timestamp']
        if len(p_times) and len(t_times):
            if p_times.max() >= t_times.min():
                ordering_violations += 1

    if ordering_violations > 0:
        print(f"⚠️  WARNING: {ordering_violations} users have timestamp ordering issues")
    else:
        print(f"✅ Timestamp ordering verified — test is always after persona")

    # ── SPLIT STATS ──────────────────────────────────────────────
    split_stats = {
        'total_users_before' : df['user_id'].nunique(),
        'users_kept'         : len(test_ids),
        'users_skipped'      : len(skipped_users),
        'persona_reviews'    : len(persona_df),
        'val_reviews'        : len(val_df),
        'test_reviews'       : len(test_df),
        'avg_persona_len'    : round(
            persona_df.groupby('user_id').size().mean(), 1
        ) if len(persona_df) else 0,
        'skipped_users'      : skipped_users
    }

    # ── PRINT SUMMARY ────────────────────────────────────────────
    print(f"{'=' * 60}")
    print(f"SPLIT SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Total users in dataset : {split_stats['total_users_before']:,}")
    print(f"  Users kept             : {split_stats['users_kept']:,}")
    print(f"  Users skipped          : {split_stats['users_skipped']:,}")
    print(f"\n  Persona set            : {split_stats['persona_reviews']:,} reviews")
    print(f"  Validation set         : {split_stats['val_reviews']:,} reviews")
    print(f"  Test set               : {split_stats['test_reviews']:,} reviews")
    print(f"\n  Avg persona length     : {split_stats['avg_persona_len']} reviews/user")
    print(f"{'=' * 60}")

    # ── WARN ABOUT TEST SET ──────────────────────────────────────
    print(f"\n🔒 TEST SET LOCKED — {len(test_df)} reviews")
    print(f"   Do NOT use test_df for prompt tuning.")
    print(f"   Use val_df during development.")
    print(f"   Run test_df ONCE at final evaluation only.\n")

    return persona_df, val_df, test_df, split_stats

In [36]:
# ── RUN IT ───────────────────────────────────────────────────────

persona_df, val_df, test_df, split_stats = split_user_reviews(
    clean_df    = rich_df,
    min_reviews = 40,
    n_test      = 1,
    n_val       = 1
)

# ── SPOT CHECK ONE USER ──────────────────────────────────────────
# verify the split looks right for a real user

sample_user = test_df['user_id'].iloc[0]

print(f"SPOT CHECK — User: {sample_user}")
print(f"{'─' * 50}")

p = persona_df[persona_df['user_id'] == sample_user]
v = val_df[val_df['user_id'] == sample_user]
t = test_df[test_df['user_id'] == sample_user]

print(f"Persona  : {len(p)} reviews | "
      f"last timestamp: {p['timestamp'].max()}")
print(f"Val      : {len(v)} reviews | "
      f"timestamp: {v['timestamp'].values}")
print(f"Test     : {len(t)} reviews | "
      f"timestamp: {t['timestamp'].values}")

# confirm ordering
assert p['timestamp'].max() < v['timestamp'].min(), \
    "❌ Persona bleeds into validation"
assert v['timestamp'].max() < t['timestamp'].min(), \
    "❌ Validation bleeds into test"

print(f"\n✅ Ordering confirmed — persona < validation < test")
print(f"\nSample persona review  : {p.iloc[-1]['text'][:100]}...")
print(f"Sample val review      : {v.iloc[0]['text'][:100]}...")
print(f"Sample test review     : {t.iloc[0]['text'][:100]}...")

TEMPORAL REVIEW SPLIT
  Min reviews required : 40
  Test holdout         : last 1 review(s) per user
  Validation holdout   : 1 review(s) before test
  Persona minimum      : 38 reviews

✅ Timestamp ordering verified — test is always after persona
SPLIT SUMMARY
  Total users in dataset : 1,364
  Users kept             : 39
  Users skipped          : 1,325

  Persona set            : 2,206 reviews
  Validation set         : 39 reviews
  Test set               : 39 reviews

  Avg persona length     : 56.6 reviews/user

🔒 TEST SET LOCKED — 39 reviews
   Do NOT use test_df for prompt tuning.
   Use val_df during development.
   Run test_df ONCE at final evaluation only.

SPOT CHECK — User: AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA
──────────────────────────────────────────────────
Persona  : 47 reviews | last timestamp: 2020-04-29 14:33:40.472000
Val      : 1 reviews | timestamp: ['2021-04-27T08:46:22.938000000']
Test     : 1 reviews | timestamp: ['2021-05-12T04:01:28.876000000']

✅ Ordering confirmed 

# Part 2 Build a Persona 


In [37]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional
from enum import Enum


# ── HELPER — reusable truncation logic ───────────────────────────

def truncate(value: str, limit: int) -> str:
    """Truncate string to limit, appending ellipsis if cut."""
    if isinstance(value, str) and len(value) > limit:
        return value[:limit - 3] + "..."
    return value


# ── ENUMS — unchanged ─────────────────────────────────────────────

class RatingTendency(str, Enum):
    generous  = "generous"
    harsh     = "harsh"
    balanced  = "balanced"

class Consistency(str, Enum):
    consistent = "consistent"
    variable   = "variable"
    polarised  = "polarised"

class WritingStyle(str, Enum):
    verbose  = "verbose"
    moderate = "moderate"
    terse    = "terse"

class PriceSensitivity(str, Enum):
    very_high = "very high"
    high      = "high"
    moderate  = "moderate"
    low       = "low"

class SkepticismLevel(str, Enum):
    very_high = "very high"
    high      = "high"
    moderate  = "moderate"
    low       = "low"


# ── RATING BEHAVIOUR ──────────────────────────────────────────────

class RatingBehaviour(BaseModel):
    tendency   : RatingTendency
    consistency: Consistency
    never_gives: list[int] = Field(
        default_factory = list,
        description     = "STRICT: List only integers 1-5. e.g. [5] or []"
    )
    pattern: str = Field(
        max_length  = 200,
        description = "STRICT CONSTRAINT: Under 200 characters. Use fragments if needed."
    )

    @field_validator('pattern', mode='before')
    @classmethod
    def truncate_pattern(cls, v):
        return truncate(str(v), 200) if v else v

    @field_validator('never_gives', mode='before')
    @classmethod
    def validate_never_gives(cls, v):
        if not isinstance(v, list):
            return []
        return [int(x) for x in v if str(x).strip().isdigit() and 1 <= int(x) <= 5]


# ── WRITING VOICE ─────────────────────────────────────────────────

class WritingVoice(BaseModel):
    style    : WritingStyle
    tone     : str = Field(
        max_length  = 150,
        description = "STRICT CONSTRAINT: Under 150 characters. Single descriptive phrase."
    )
    structure: str = Field(
        max_length  = 250,
        description = "STRICT CONSTRAINT: Under 250 characters. How they organise a review."
    )
    signature_phrases: list[str] = Field(
        default_factory = list,
        description     = "Actual phrases from their reviews. No invented phrases."
    )

    @field_validator('tone', mode='before')
    @classmethod
    def truncate_tone(cls, v):
        return truncate(str(v), 150) if v else v

    @field_validator('structure', mode='before')
    @classmethod
    def truncate_structure(cls, v):
        return truncate(str(v), 250) if v else v

    @field_validator('signature_phrases', mode='before')
    @classmethod
    def clean_phrases(cls, v):
        if not isinstance(v, list):
            return []
        # truncate any individual phrase that is too long
        return [str(p)[:100] for p in v if p]


# ── NIGERIAN SIGNALS ──────────────────────────────────────────────

class NigerianSignals(BaseModel):
    price_sensitivity  : PriceSensitivity
    scepticism         : SkepticismLevel
    community_oriented : bool
    cultural_notes     : Optional[str] = Field(
        default     = None,
        max_length  = 1000,
        description = "STRICT CONSTRAINT: Under 1000 characters. Nigerian consumer signals observed."
    )

    @field_validator('cultural_notes', mode='before')
    @classmethod
    def truncate_cultural_notes(cls, v):
        if v is None:
            return v
        return truncate(str(v), 1000)

    @field_validator('community_oriented', mode='before')
    @classmethod
    def coerce_bool(cls, v):
        # handle cases where LLM returns "true"/"false" as strings
        if isinstance(v, str):
            return v.lower() in ('true', 'yes', '1')
        return bool(v)


# ── PERSONA DOSSIER ───────────────────────────────────────────────

class PersonaDossier(BaseModel):
    """
    Agent 1 output — the inter-agent contract.
    Structural fields validated strictly.
    Discovery fields free-form with graceful truncation.
    """
    user_id      : str
    core_identity: str = Field(
        max_length  = 600,
        description = "STRICT CONSTRAINT: Under 600 characters. Who this person is as a reviewer."
    )
    rating_behaviour    : RatingBehaviour
    writing_voice       : WritingVoice
    deep_traits         : list[str] = Field(
        min_length  = 3,
        description = "Specific discovered traits. Each must be a full observation, not a single word."
    )
    what_they_care_about: list[str] = Field(
        min_length  = 1,
        description = "Ranked by importance — most dominant first."
    )
    what_they_ignore    : list[str] = Field(default_factory=list)
    context_clues       : Optional[str] = Field(
        default     = None,
        max_length  = 1000,
        description = "STRICT CONSTRAINT: Under 1000 characters. Life context revealed in reviews."
    )
    nigerian_signals    : NigerianSignals
    simulation_brief    : str = Field(
        max_length  = 1000,
        description = (
            "STRICT CONSTRAINT: Under 1000 characters. "
            "Written directly to the generation model. "
            "How to sound like this person — voice, priorities, quirks."
        )
    )

    # computed fields — injected after LLM call
    avg_rating  : float = 3.0
    rating_std  : float = 0.0
    review_count: int   = 0

    # ── TRUNCATION VALIDATORS ─────────────────────────────────────

    @field_validator('core_identity', mode='before')
    @classmethod
    def truncate_core_identity(cls, v):
        return truncate(str(v), 600) if v else v

    @field_validator('context_clues', mode='before')
    @classmethod
    def truncate_context_clues(cls, v):
        if v is None:
            return v
        return truncate(str(v), 1000)

    @field_validator('simulation_brief', mode='before')
    @classmethod
    def truncate_simulation_brief(cls, v):
        return truncate(str(v), 1000) if v else v

    @field_validator('deep_traits', mode='before')
    @classmethod
    def clean_deep_traits(cls, v):
        if not isinstance(v, list):
            return []
        # truncate any individual trait that is excessively long
        # but don't filter content — agent decides what's relevant
        return [str(t)[:300] for t in v if t and len(str(t).strip()) > 3]

    @field_validator('what_they_care_about', mode='before')
    @classmethod
    def clean_cares_about(cls, v):
        if not isinstance(v, list):
            return ['product quality']
        return [str(t)[:200] for t in v if t]

    @field_validator('what_they_ignore', mode='before')
    @classmethod
    def clean_ignores(cls, v):
        if not isinstance(v, list):
            return []
        return [str(t)[:200] for t in v if t]

    # ── STRUCTURAL VALIDATORS — kept strict ───────────────────────

    @field_validator('avg_rating')
    @classmethod
    def valid_rating_range(cls, v):
        if not 1.0 <= v <= 5.0:
            raise ValueError(f"avg_rating {v} out of range 1–5")
        return round(v, 2)

    @field_validator('rating_behaviour')
    @classmethod
    def valid_never_gives(cls, v):
        for star in v.never_gives:
            if star not in [1, 2, 3, 4, 5]:
                raise ValueError(f"never_gives contains invalid value: {star}")
        return v

    # ── MODEL VALIDATOR — cross-field check ───────────────────────

    @model_validator(mode='after')
    def brief_references_identity(self):
        """
        Soft check — simulation_brief should not be empty
        if we have enough identity information to fill it.
        Falls back gracefully rather than raising.
        """
        if not self.simulation_brief or len(self.simulation_brief) < 20:
            self.simulation_brief = (
                f"Write as a {self.rating_behaviour.tendency} rater "
                f"with a {self.writing_voice.tone} tone. "
                f"They care most about: {', '.join(self.what_they_care_about[:2])}."
            )
        return self


# ── SIMULATED REVIEW ──────────────────────────────────────────────

class SimulatedReview(BaseModel):
    """
    Agent 2 output — the final deliverable.
    """
    user_id  : str
    asin     : str
    reasoning: str = Field(
        max_length  = 800,
        description = "STRICT CONSTRAINT: Under 800 characters."
    )
    rating  : int  = Field(ge=1, le=5)
    title   : str  = Field(max_length=200)
    review  : str  = Field(
        min_length  = 10,
        description = "Full review in the user's voice."
    )
    language: str = "english"

    @field_validator('reasoning', mode='before')
    @classmethod
    def truncate_reasoning(cls, v):
        return truncate(str(v), 800) if v else v

    @field_validator('title', mode='before')
    @classmethod
    def truncate_title(cls, v):
        return truncate(str(v), 200) if v else v

    @field_validator('rating', mode='before')
    @classmethod
    def coerce_and_clamp_rating(cls, v):
        # handle "4 stars", "4/5", "4.5" etc
        if isinstance(v, str):
            match = re.search(r'\d+', v)
            v = int(match.group()) if match else 3
        return max(1, min(5, int(float(v))))

    @field_validator('review', mode='before')
    @classmethod
    def review_not_placeholder(cls, v):
        placeholders = {'n/a', 'none', 'null', '[review]', 'review text', 'placeholder'}
        if not v or str(v).lower().strip() in placeholders:
            raise ValueError("Review is a placeholder — generation failed")
        return str(v)

In [38]:
# instantiating my models 
client = genai.Client(api_key=GOOGLE_API_KEY)

# model names — pass these as strings into your agent functions
FLASH_MODEL = "gemini-3.1-flash-lite" #"gemini-2.5-flash-lite"
PRO_MODEL   = "gemini-3.1-flash-lite" 

In [39]:
# ── CACHE LAYER ──────────────────────────────────────────────────

class PersonaCache:
    """
    Stores Agent 1 outputs to disk.
    Same user never costs an API call twice.
    """
    def __init__(self, cache_dir='/kaggle/working/persona_cache'):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def _key(self, user_id):
        return os.path.join(
            self.cache_dir,
            f"{hashlib.md5(user_id.encode()).hexdigest()}.json"
        )

    def get(self, user_id):
        path = self._key(user_id)
        if os.path.exists(path):
            with open(path, 'r') as f:
                return json.load(f)
        return None

    def set(self, user_id, profile):
        with open(self._key(user_id), 'w') as f:
            json.dump(profile, f, indent=2)

    def exists(self, user_id):
        return os.path.exists(self._key(user_id))

    def size(self):
        return len(os.listdir(self.cache_dir))


cache = PersonaCache()


# ── TOKEN SAVING UTILITIES ───────────────────────────────────────

def compress_review(text, max_words=80):
    """
    Trims a review to max_words while keeping meaning.
    Removes excessive punctuation and whitespace.
    Saves tokens without losing signal.
    """
    if not text:
        return ""
    # collapse whitespace
    text = re.sub(r'\s+', ' ', str(text).strip())
    # remove repeated punctuation
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[.]{2,}', '...', text)
    words = text.split()
    if len(words) <= max_words:
        return text
    # truncate but end at a sentence boundary if possible
    truncated = ' '.join(words[:max_words])
    last_stop = max(
        truncated.rfind('.'),
        truncated.rfind('!'),
        truncated.rfind('?')
    )
    if last_stop > max_words * 3:   # only use boundary if it's not too short
        return truncated[:last_stop + 1]
    return truncated + '...'


def select_representative_reviews(texts, ratings, n=6):
    """
    Picks the most signal-rich reviews to send to Agent 1.
    Selects across the rating spectrum — not just recent ones.
    This gives Agent 1 a balanced picture in fewer tokens.

    Strategy:
    - 1 lowest rated review  (reveals complaints)
    - 1 highest rated review (reveals praise style)  
    - 1 middle rated review  (reveals nuance)
    - 3 most recent reviews  (reveals current voice)
    """
    if not texts or len(texts) == 0:
        return []

    paired    = list(zip(texts, ratings))
    selected  = []
    seen_idx  = set()

    # lowest rating
    min_idx = min(range(len(paired)), key=lambda i: paired[i][1])
    selected.append(paired[min_idx])
    seen_idx.add(min_idx)

    # highest rating
    max_idx = max(range(len(paired)), key=lambda i: paired[i][1])
    if max_idx not in seen_idx:
        selected.append(paired[max_idx])
        seen_idx.add(max_idx)

    # middle rating (closest to 3)
    mid_idx = min(
        [i for i in range(len(paired)) if i not in seen_idx],
        key=lambda i: abs(paired[i][1] - 3),
        default=None
    )
    if mid_idx is not None:
        selected.append(paired[mid_idx])
        seen_idx.add(mid_idx)

    # most recent (last n reviews, skip already selected)
    for i in range(len(paired) - 1, -1, -1):
        if i not in seen_idx and len(selected) < n:
            selected.append(paired[i])
            seen_idx.add(i)

    return selected


def build_compact_history(texts, ratings, max_words_per_review=80, n_reviews=6):
    """
    Builds a token-efficient review history block for Agent 1.
    Selects representative reviews and compresses each one.
    """
    selected = select_representative_reviews(texts, ratings, n=n_reviews)
    lines    = []
    for i, (text, rating) in enumerate(selected):
        compressed = compress_review(text, max_words=max_words_per_review)
        lines.append(f"[{rating}★] {compressed}")
    return "\n".join(lines)


# ── AGENT 1 WITH PYDANTIC ────────────────────────────────────────

def run_agent1_analyst(
    user_id    : str,
    texts      : list,
    ratings    : list,
    flash_model: str,        # model name string e.g. "gemini-2.5-flash"
    max_retries: int = 2
) -> 'PersonaDossier':
 
    # ── CACHE CHECK ───────────────────────────────────────────────
    cached = cache.get(user_id)
    if cached:
        try:
            return PersonaDossier(**cached)
        except Exception:
            pass    # stale — re-run
 
    # ── COMPUTE STATS ─────────────────────────────────────────────
    avg_rating = round(sum(ratings) / len(ratings), 2) if ratings else 3.0
    rating_std = round(
        (sum((r - avg_rating) ** 2 for r in ratings) / len(ratings)) ** 0.5, 2
    ) if ratings else 0.0
 
    history_block = build_compact_history(texts, ratings)
    stats_block   = (
        f"Total: {len(ratings)} reviews | "
        f"Avg: {avg_rating}/5 | "
        f"Std: {rating_std} | "
        f"Spread: {sorted(set(ratings))}"
    )
 
    # schema hint tells the model exactly what structure to return
    schema_hint = json.dumps(PersonaDossier.model_json_schema(), indent=2)
 
    prompt = f"""Analyse this Amazon reviewer and produce their persona dossier.
Discover traits dynamically — do not use generic categories.
Be concise overall. For deep_traits specifically, write each as a full observable phrase — these will be used for keyword matching.
Never exceed the character limits in the schema.
Front-load the most important information.
 
STATS: {stats_block}
 
REVIEWS:
{history_block}
 
Return ONLY valid JSON matching this exact schema:
{schema_hint}
 
Rules:
- deep_traits must be specific observations, not generic single words
- simulation_brief is written directly to the generation model
- never_gives is a list of integers e.g. [5] or []
- No markdown, no explanation, JSON only"""
 
    # ── GENERATION WITH MANUAL JSON PARSING ───────────────────────
    for attempt in range(max_retries + 1):
        try:
            response = client.models.generate_content(
                model    = flash_model,
                contents = prompt,
                config   = types.GenerateContentConfig(
                    temperature       = 0.3,
                    max_output_tokens = 1500,
                )
            )
 
            raw = response.text.strip()
 
            # strip markdown fences if model wraps in ```json ... ```
            raw = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
            raw = re.sub(r'^```\s*',     '', raw, flags=re.MULTILINE)
            raw = re.sub(r'\s*```$',     '', raw, flags=re.MULTILINE)
 
            data = json.loads(raw)
 
            # inject computed stats — LLM doesn't know these
            data['user_id']      = user_id
            data['avg_rating']   = avg_rating
            data['rating_std']   = rating_std
            data['review_count'] = len(ratings)
 
            # Pydantic validates the full structure + runs all validators
            dossier = PersonaDossier(**data)
 
            cache.set(user_id, dossier.model_dump())
            return dossier
 
        except json.JSONDecodeError as e:
            print(f"  ⚠️  Agent 1 attempt {attempt + 1} — JSON parse failed: {e}")
        except ValueError as e:
            print(f"  ⚠️  Agent 1 attempt {attempt + 1} — Validation failed: {e}")
        except Exception as e:
            print(f"  ⚠️  Agent 1 attempt {attempt + 1} — {type(e).__name__}: {e}")
 
    # ── FALLBACK ──────────────────────────────────────────────────
    print(f"  ❌ All Agent 1 attempts failed for {user_id} — using fallback")
    return PersonaDossier(
        user_id          = user_id,
        core_identity    = "Reviewer with insufficient profile data.",
        rating_behaviour = RatingBehaviour(
            tendency    = RatingTendency.balanced,
            consistency = Consistency.consistent,
            never_gives = [],
            pattern     = "No pattern detected."
        ),
        writing_voice = WritingVoice(
            style             = WritingStyle.moderate,
            tone              = "neutral",
            structure         = "standard",
            signature_phrases = []
        ),
        deep_traits          = [
            "No distinctive traits detected from available history",
            "Review history too sparse for deep profiling",
            "Defaults to standard reviewer behaviour",
            "No strong category preferences detected"
        ],
        what_they_care_about = ["product quality", "value for money"],
        what_they_ignore     = [],
        nigerian_signals     = NigerianSignals(
            price_sensitivity  = PriceSensitivity.moderate,
            scepticism         = SkepticismLevel.moderate,
            community_oriented = False,
            cultural_notes     = None
        ),
        simulation_brief = (
            f"Write a balanced review consistent with a {avg_rating:.1f}-star "
            f"average rater. Keep tone neutral and length moderate."
        ),
        avg_rating   = avg_rating,
        rating_std   = rating_std,
        review_count = len(ratings)
    )


In [40]:
#------------Grounding search for the product description---------------------
# adding Grounding search for the product description helps the agent2 know what its reviewing 


class ItemCard(BaseModel):
    asin           : str = ""
    title          : str = ""
    product_summary: str = Field(
        max_length  = 400,
        description = "2-3 sentence plain English description of what this product is"
    )
    key_features   : list[str] = Field(
        default_factory = list,
        description     = "Up to 5 specific product features"
    )
    typical_use_case: str = Field(
        max_length  = 200,
        description = "Who uses this and why"
    )
    price_tier     : str = Field(
        default     = "mid-range",
        description = "budget | mid-range | premium"
    )
    brand_notes    : str = Field(
        default     = "",
        max_length  = 200,
        description = "Anything notable about the brand, empty string if unknown"
    )
    
    # silently truncate instead of raising — LLM often writes long strings
    @field_validator('product_summary', mode='before')
    @classmethod
    def trim_summary(cls, v):
        return str(v)[:400] if v else ""
 
    @field_validator('typical_use_case', mode='before')
    @classmethod
    def trim_use_case(cls, v):
        return str(v)[:200] if v else ""
 
    @field_validator('brand_notes', mode='before')
    @classmethod
    def trim_brand_notes(cls, v):
        return str(v)[:200] if v else ""
 
    @field_validator('price_tier', mode='before')
    @classmethod
    def normalise_price_tier(cls, v):
        # model sometimes returns "mid range", "Mid-Range", "unknown" etc.
        v = str(v).lower().strip()
        if 'budget' in v or 'low' in v:
            return 'budget'
        if 'premium' in v or 'high' in v or 'luxury' in v:
            return 'premium'
        return 'mid-range'
 
    @field_validator('key_features', mode='before')
    @classmethod
    def clean_features(cls, v):
        if not isinstance(v, list):
            return []
        return [str(f)[:100] for f in v if f][:5]   # max 5, max 100 chars each



# ── ITEM ENRICHMENT ──────────────────────────────────────────────
 
def enrich_item_card(
    asin        : str,
    title       : str,
    description : str,      # existing description_text — may be empty
    category    : str,
    flash_model : str,
    cache_dir   : str = '/kaggle/working/item_cache'
) -> ItemCard:
 
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{asin}.json")
 
    # ── CACHE CHECK — same ASIN never searched twice ───────────────
    if os.path.exists(cache_path):
        try:
            with open(cache_path) as f:
                return ItemCard(**json.load(f))
        except Exception:
            pass    # corrupted cache — re-fetch
 
    prompt = f"""Search for this product and return a structured description.
 
Product title    : {title}
Category         : {category}
Existing description (may be empty): {description[:200]}
 
Return a JSON object with exactly these fields:
{{
  "product_summary"  : "<2-3 sentence plain English description of what this product is>",
  "key_features"     : ["<feature 1>", "<feature 2>", "<feature 3>"],
  "typical_use_case" : "<who uses this and why, one sentence>",
  "price_tier"       : "<exactly one of: budget | mid-range | premium>",
  "brand_notes"      : "<one sentence about the brand, or empty string>"
}}"""
 
    item_card = None
 
    # ── SINGLE ATTEMPT: google_search + manual json parse ─────────
    # response_mime_type is intentionally omitted — the API rejects
    # tool use + response_mime_type together (400 INVALID_ARGUMENT).
    try:
        response = client.models.generate_content(
            model    = flash_model,
            contents = prompt,
            config   = types.GenerateContentConfig(
                tools       = [types.Tool(google_search=types.GoogleSearch())],
                temperature = 0.1,
            )
        )
 
        raw = response.text.strip()
        raw = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
        raw = re.sub(r'^```\s*',     '', raw, flags=re.MULTILINE)
        raw = re.sub(r'\s*```$',     '', raw, flags=re.MULTILINE)
 
        data      = json.loads(raw)
        item_card = ItemCard(**data)    # validators truncate long strings
 
    except Exception as e:
        print(f"  ⚠️  ItemCard enrichment failed ({e}) — using metadata fallback")
 
    # ── FALLBACK ──────────────────────────────────────────────────
    if item_card is None:
        item_card = ItemCard(
            product_summary  = description[:300] if description else f"{category} product: {title}",
            key_features     = [],
            typical_use_case = category,
            price_tier       = "mid-range",
            brand_notes      = ""
        )
 
    # inject identifiers
    item_card.asin  = asin
    item_card.title = title
 
    # ── CACHE ─────────────────────────────────────────────────────
    with open(cache_path, 'w') as f:
        json.dump(item_card.model_dump(), f, indent=2)
 
    return item_card
 

In [41]:
# ── OPTIMIZED AGENT 2 WITH TWO-STAGE EXPLICIT REASONING CHAIN ──

def run_agent2_with_rag(
    dossier          : 'PersonaDossier',
    item_asin        : str,
    item_title       : str,
    item_description : str,
    item_category    : str,
    pro_model        : str,
    index            : 'faiss.Index',
    metadata         : list,
    embedding_model  : 'SentenceTransformer',
    nigerian_language: Optional[str] = None,
    top_k            : int = 4,
    max_retries      : int = 2,
    item_card        : 'ItemCard | None' = None
) -> 'SimulatedReview':
 
    # ── RAG RETRIEVAL ─────────────────────────────────────────────
    retrieved  = retrieve_for_agent2(
        user_id             = dossier.user_id,
        item_title          = item_title,
        item_description    = item_description,
        item_category       = item_category,
        index               = index,
        metadata            = metadata,
        embedding_model     = embedding_model,
        top_k               = top_k,
        same_category_boost = True
    )
    rag_block  = format_rag_examples(retrieved)
    n_same_cat = sum(1 for r in retrieved if r.get('same_category'))

    # ── PRODUCT BLOCK — enriched if item_card available ───────────
    if item_card is not None:
        product_block = f"""Title         : {item_card.title}
        ASIN          : {item_asin}
        Category      : {item_category}
        What it is    : {item_card.product_summary}
        Key features  : {' | '.join(item_card.key_features) or 'not available'}
        Typical use   : {item_card.typical_use_case}
        Price tier    : {item_card.price_tier}
        Brand notes   : {item_card.brand_notes or 'none'}"""
    else:
        product_block = f"""Title      : {item_title}
        ASIN       : {item_asin}
        Category   : {item_category}
        Description: {str(item_description)[:400]}"""
 
    # ── MATHEMATICAL PROFILE WINDOW (CRITICAL FOR RMSE) ───────────
    user_mean = dossier.avg_rating
    user_std  = dossier.rating_std if dossier.rating_std > 0 else 0.5
    
    # Calculate a rigid 2-standard-deviation bounding window
    lower_bound = max(1, round(user_mean - (2 * user_std)))
    upper_bound = min(5, round(user_mean + (2 * user_std)))
 
    # ── NIGERIAN CONDITIONING ─────────────────────────────────────
    lang_instructions = {
        'pidgin'           : "Nigerian Pidgin — weave in naturally: 'e good o', 'I no go lie', 'wahala', 'sha'",
        'yoruba_influenced': "Yoruba-influenced English — 'omo', 'my people', occasional Yoruba words",
        'igbo_influenced'  : "Igbo-influenced English — 'nna', 'chai', direct assertive tone",
        'hausa_influenced' : "Hausa-influenced English — 'wallahi', 'alhamdulillah', respectful tone",
    }
    lang_line = ""
    if nigerian_language and nigerian_language in lang_instructions:
        lang_line = f"\nLANGUAGE STYLE: {lang_instructions[nigerian_language]}\\n"
 
    # ── STEP 1: RATING ESTIMATION WITH FRONT-LOADED REASONING ──
    # CoT reasoning MUST appear before the raw rating key to ground attention
    committed_rating = round(user_mean)    
    committed_reason = "Fallback — calculation window defaulted to historical mean."
 
    rating_prompt = f"""You are analyzing a product to determine what star rating this unique reviewer profile would assign.
    
CRITICAL PROFILE CONSTRAINTS:
- Reviewer baseline average rating: {user_mean:.2f} / 5 stars
- Reviewer baseline variance deviation: {user_std:.2f}
- STALWART STATISTICAL LIMITS: Your final choice MUST stay strictly within {lower_bound} to {upper_bound} stars based on their lifetime history.
- Never gives options: {dossier.rating_behaviour.never_gives or 'none'}

PRODUCT CONTEXT:
{product_block}
 
PAST USER EXPERIENCES FOR COMPARISON:
{rag_block}
 
OUTPUT DIRECTIONS:
Evaluate how the product highlights map to what this persona cares about. 
Write your analytical justification FIRST, then conclude with the numerical rating token.

Return ONLY a raw JSON object structured exactly like this:
{{
  "reasoning": "<write your contextual behavior analysis here under 200 characters>",
  "rating": <provide a single integer within the user's hard bounds of {lower_bound} to {upper_bound}>
}}
No markdown formatting fences. No alternative fields."""
 
    try:
        r1 = client.models.generate_content(
            model    = pro_model,
            contents = rating_prompt,
            config   = types.GenerateContentConfig(
                temperature       = 0.1,  # Low temperature forces mathematical constraint adherence
                max_output_tokens = 250,
            )
        )
        raw1 = r1.text.strip()
        raw1 = re.sub(r'^```json\s*', '', raw1, flags=re.MULTILINE)
        raw1 = re.sub(r'^```\s*',     '', raw1, flags=re.MULTILINE)
        raw1 = re.sub(r'\s*```$',     '', raw1, flags=re.MULTILINE)
        
        d1 = json.loads(raw1.strip())
        
        # Apply programmatic bounding-box safety rails over raw LLM values
        parsed_rating = int(float(d1.get('rating', committed_rating)))
        committed_rating = max(lower_bound, min(upper_bound, parsed_rating))
        committed_reason = str(d1.get('reasoning', committed_reason))[:200]
        
    except Exception as e:
        print(f"  ⚠️  Rating estimation pipeline fell back to safe window limits ({e})")
        committed_rating = max(lower_bound, min(upper_bound, round(user_mean)))
 
    # ── STEP 2: GENERATE REVIEW CONDITIONAL ON CLAMPED RATING ──
    output_example = (
        '{\n'
        '  "reasoning": "' + committed_reason.replace('"', '\\"') + '",\n'
        '  "rating": ' + str(committed_rating) + ',\n'
        '  "title": "Enter short title here",\n'
        '  "review": "Enter generated body review text here matching the tone instructions.",\n'
        '  "user_id": "' + dossier.user_id + '",\n'
        '  "asin": "' + item_asin + '",\n'
        '  "language": "english"\n'
        '}'
    )
 
    review_prompt = f"""You are generating the final text review matching a specific consumer's voice.
The final rating is permanently locked at exactly: {committed_rating}★
Your generated text body must be completely coherent with a {committed_rating}★ evaluation.

REVIEWER VOICE PARAMETERS:
{dossier.simulation_brief}
Identity Profile: {dossier.core_identity}
Tone & Cadence   : {dossier.writing_voice.tone} | {dossier.writing_voice.style}
Signature Phrases: {', '.join(dossier.writing_voice.signature_phrases) or 'none found'}

PAST STRUCTURAL TEXT EXAMPLES:
{rag_block}
{lang_line}
PRODUCT METADATA CARD:
{product_block}

ANTI-DRIFT RUNTIME INSTRUCTIONS:
1. Mirror the historical length pattern shown in the past structural text examples. Do not write filler.
2. Weave the following traits naturally into the narrative without referencing them directly:
{chr(10).join(f'   - {t}' for t in dossier.deep_traits)}
3. Embody the exact level of critical nuance required to justify a score of {committed_rating}★.

OUTPUT RULES:
- Return ONLY a single JSON object.
- The "rating" key must appear before the "review" body key.
- No markdown fences or formatting characters outside the json structure.

MATCH THIS TARGET STRUCTURAL FORMAT:
{output_example}
"""
 
    for attempt in range(max_retries + 1):
        try:
            response = client.models.generate_content(
                model    = pro_model,
                contents = review_prompt,
                config   = types.GenerateContentConfig(
                    temperature       = 0.6,  # Allows creative voice mimicry while anchored to the rating
                    max_output_tokens = 800,
                )
            )
 
            raw = response.text.strip()
            raw = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
            raw = re.sub(r'^```\s*',     '', raw, flags=re.MULTILINE)
            raw = re.sub(r'\s*```$',     '', raw, flags=re.MULTILINE)
            raw = raw.strip()
 
            brace_count = 0
            end_pos     = 0
            for i, char in enumerate(raw):
                if char == '{':
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        end_pos = i + 1
                        break
            if end_pos > 0:
                raw = raw[:end_pos]
 
            data = json.loads(raw)
 
            data['user_id']  = dossier.user_id
            data['asin']     = item_asin
            data['language'] = nigerian_language or 'english'
            data['rating']   = committed_rating
            data['reasoning'] = committed_reason
 
            return SimulatedReview(**data)
 
        except Exception as e:
            print(f"  ⚠️  Agent 2 text layout synthesis attempt {attempt + 1} failed: {e}")
 
    # ── CRITICAL RECOVERY FALLBACK ────────────────────────────────
    return SimulatedReview(
        user_id   = dossier.user_id,
        asin      = item_asin,
        reasoning = committed_reason,
        rating    = committed_rating,
        title     = f"Review for {item_title[:30]}",
        review    = f"This item is a standard functional option. It matches expectations for this segment.",
        language  = nigerian_language or 'english'
    )

In [42]:
# ── MASTER PIPELINE ───────────────────────────────────────────────

def simulate_review_two_agent(
    user_id          : str,
    texts            : list,
    ratings          : list,
    item_asin        : str,
    item_title       : str,
    item_description : str,
    item_category    : str,          # ← added — was missing, caused holdout bug
    flash_model,
    pro_model,
    rag_index,                       # ← added — no longer hardcoded global
    rag_metadata     : list,         # ← added
    embedding_model,                 # ← added
    nigerian_language: Optional[str] = None,
    top_k            : int           = 4,
    verbose          : bool          = False
) -> tuple[PersonaDossier, SimulatedReview]:
    """
    Full two-agent pipeline with RAG grounding.

    Agent 1 (Flash)  — analyses history, builds dossier, cached
    Agent 2 (Pro)    — generates review from dossier + RAG examples

    Parameters:
        user_id           : reviewer ID
        texts             : list of their past review texts
        ratings           : list of their past ratings (same order)
        item_asin         : product ASIN
        item_title        : product title
        item_description  : product description
        item_category     : product category — used for RAG retrieval boost
        flash_model       : Gemini Flash instance (Agent 1)
        pro_model         : Gemini Pro instance (Agent 2)
        rag_index         : loaded FAISS index
        rag_metadata      : parallel metadata list
        embedding_model   : loaded SentenceTransformer
        nigerian_language : optional cultural conditioning
        top_k             : number of RAG examples to retrieve
        verbose           : print step-by-step progress
    """

    # ── AGENT 1 ──────────────────────────────────────────────────
    if verbose:
        hit = cache.exists(user_id)
        print(f"\n[Agent 1] {'Cache hit ✅' if hit else 'Analysing with Flash...'}")

    dossier = run_agent1_analyst(
        user_id     = user_id,
        texts       = texts,
        ratings     = ratings,
        flash_model = FLASH_MODEL
    )

    if verbose:
        print(f"  Traits found : {len(dossier.deep_traits)}")
        print(f"  Identity     : {dossier.core_identity[:70]}...")

# ── ITEM ENRICHMENT — product context ─────────────────────────
    if verbose:
        print(f"[Enrichment] Looking up product: {item_title[:50]}...")
 
    item_card = enrich_item_card(
        asin        = item_asin,
        title       = item_title,
        description = item_description,
        category    = item_category,
        flash_model = flash_model
    )
 
    if verbose:
        print(f"  Summary      : {item_card.product_summary[:80]}...")

    # ── AGENT 2 WITH RAG ─────────────────────────────────────────
    print(f"[Agent 2] Generating with RAG + Pro...")
    review = run_agent2_with_rag(
        dossier           = dossier,
        item_asin         = item_asin,
        item_title        = item_title,
        item_description  = item_description,
        item_category     = item_category,    # ← clean — no holdout reference
        pro_model         = PRO_MODEL,
        index             = rag_index,
        metadata          = rag_metadata,
        embedding_model   = embedding_model,
        item_card         = item_card,
        nigerian_language = nigerian_language,
        top_k             = top_k
    )

    if verbose:
        print(f"  Rating       : {review.rating}★")
        print(f"  Language     : {review.language}")

    return dossier, review

### RAG Implementation 
cell above cannot run without the rag cell so run it first or change the arriangement

In [43]:
#this function is needed below
def safe_timestamp(value) -> float:
    """
    Converts any timestamp format to a float unix timestamp.
    Handles: int, float, pd.Timestamp, datetime, string, None
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return 0.0
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, pd.Timestamp):
        return float(value.timestamp())
    if hasattr(value, 'timestamp'):          # any datetime-like object
        return float(value.timestamp())
    try:
        return float(pd.Timestamp(value).timestamp())
    except Exception:
        return 0.0



# ── CONSTANTS ────────────────────────────────────────────────────
RAG_INDEX_PATH   = '/kaggle/working/rag_index.faiss'
RAG_META_PATH    = '/kaggle/working/rag_meta.pkl'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'   # fast, lightweight, strong for semantic similarity

# ── SET HUGGINGFACE TOKEN ────────────────────────────────────────
HF_TOKEN = HF_Key
os.environ["HF_TOKEN"] = HF_TOKEN


# ── STEP 1: BUILD THE RAG INDEX ───────────────────────────────────

def build_rag_index(persona_df: pd.DataFrame, embedding_model: SentenceTransformer):
    """
    Builds a FAISS vector index over all reviews in persona_df.
    Run once during setup — saved to disk and reloaded on demand.

    Each review is stored with its:
    - embedding vector  (for similarity search)
    - user_id           (so we only retrieve from the right user)
    - review text       (what gets passed to Agent 2)
    - rating            (for context)
    - category          (for category-aware retrieval)

    Parameters:
        persona_df      : your persona set from split_user_reviews()
        embedding_model : loaded SentenceTransformer instance

    Returns:
        index    : FAISS index
        metadata : list of dicts, one per review — parallel to index
    """

    print("Building RAG index...")
    print(f"  Reviews to index: {len(persona_df):,}")

    texts    = persona_df['text'].fillna('').tolist()
    metadata = []

    for _, row in persona_df.iterrows():
        metadata.append({
            'user_id'   : row.get('user_id', ''),
            'text'      : str(row.get('text', '')),
            'rating'    : float(row.get('rating', 3.0)),
            'category'  : str(row.get('main_category', '')),
            'asin'      : str(row.get('parent_asin', '')),
            'timestamp' : safe_timestamp(row.get('timestamp', 0)),
        })

    # encode all reviews in batches — memory efficient
    print("  Encoding reviews...")
    embeddings = embedding_model.encode(
        texts,
        batch_size      = 64,
        show_progress_bar = True,
        convert_to_numpy  = True,
        normalize_embeddings = True    # normalise for cosine similarity
    )

    # build flat FAISS index — exact search, good for this dataset size
    dimension = embeddings.shape[1]
    index     = faiss.IndexFlatIP(dimension)   # inner product = cosine on normalised vectors
    index.add(embeddings.astype(np.float32))

    # save to disk
    faiss.write_index(index, RAG_INDEX_PATH)
    with open(RAG_META_PATH, 'wb') as f:
        pickle.dump(metadata, f)

    print(f"  Index built: {index.ntotal:,} vectors, dimension {dimension}")
    print(f"  Saved to {RAG_INDEX_PATH}")

    return index, metadata


# ── STEP 2: LOAD THE RAG INDEX ────────────────────────────────────

def load_rag_index():
    """
    Loads the FAISS index and metadata from disk.
    Call this at evaluation time instead of rebuilding.
    """
    if not os.path.exists(RAG_INDEX_PATH):
        raise FileNotFoundError(
            f"RAG index not found at {RAG_INDEX_PATH}. "
            f"Run build_rag_index() first."
        )

    index = faiss.read_index(RAG_INDEX_PATH)
    with open(RAG_META_PATH, 'rb') as f:
        metadata = pickle.load(f)

    print(f"RAG index loaded: {index.ntotal:,} vectors")
    return index, metadata


# ── STEP 3: RETRIEVE FOR AGENT 2 ─────────────────────────────────

def retrieve_for_agent2(
    user_id             : str,
    item_title          : str,
    item_description    : str,
    item_category       : str,
    index               : faiss.Index,
    metadata            : list,
    embedding_model     : SentenceTransformer,
    top_k               : int  = 4,
    same_category_boost : bool = True
) -> list[dict]:

    # ── BUILD PRODUCT QUERY ───────────────────────────────────────
    query_parts = [p for p in [item_title, item_description, item_category] if p]
    query       = ' '.join(query_parts)[:512]

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True
    ).astype(np.float32)

    # ── SEARCH FULL INDEX ─────────────────────────────────────────
    search_k        = index.ntotal   # always search everything
    scores, indices = index.search(query_embedding, search_k)
    scores          = scores[0]
    indices         = indices[0]

    # ── FILTER TO THIS USER ───────────────────────────────────────
    user_results = []

    for score, idx in zip(scores, indices):
        if idx < 0 or idx >= len(metadata):
            continue

        record = metadata[idx]

        if record['user_id'] != user_id:
            continue

        if len(record['text'].split()) < 10:
            continue

        user_results.append({
            'text'            : record['text'],
            'rating'          : record['rating'],
            'category'        : record['category'],
            'asin'            : record['asin'],
            'timestamp'       : record.get('timestamp', 0),
            'similarity_score': float(score),
            'same_category'   : (
                record['category'].lower() == item_category.lower()
                if item_category else False
            ),
            'source'          : 'rag'
        })

    # ── CATEGORY BOOST ────────────────────────────────────────────
    if same_category_boost:
        for r in user_results:
            if r['same_category']:
                r['similarity_score'] += 0.15

    # ── RANK ──────────────────────────────────────────────────────
    user_results.sort(key=lambda x: x['similarity_score'], reverse=True)

    # ── CHECK FOR SAME-CATEGORY RESULTS ──────────────────────────
    # if top results have no same-category match at all,
    # trigger fallback to latest reviews instead
    same_cat_found = any(r['same_category'] for r in user_results[:top_k])

    if user_results and same_cat_found:
        # happy path — semantic results include category match
        return user_results[:top_k]

    if user_results and not same_cat_found:
        # we have results but none match the category
        # return what we have — better than nothing
        # but log it so you know
        print(
            f"  ℹ️  No same-category results for {user_id} "
            f"in '{item_category}' — returning best semantic matches"
        )
        return user_results[:top_k]

    # ── FALLBACK: no results at all — return latest reviews ───────
    print(f"  ⚠️  No RAG results for {user_id} — falling back to latest reviews")

    all_user_records = [
        {
            'text'            : m['text'],
            'rating'          : m['rating'],
            'category'        : m['category'],
            'asin'            : m['asin'],
            'timestamp'       : m.get('timestamp', 0),
            'similarity_score': 0.0,
            'same_category'   : (
                m['category'].lower() == item_category.lower()
                if item_category else False
            ),
            'source'          : 'fallback_latest'
        }
        for m in metadata
        if m['user_id'] == user_id
        and len(m['text'].split()) >= 10
    ]

    if not all_user_records:
        print(f"  ❌ User {user_id} has no usable records in metadata at all")
        return []

    # sort by timestamp — most recent first
    all_user_records.sort(key=lambda x: x['timestamp'], reverse=True)
    top_results = all_user_records[:top_k]
    print(f"  ✅ Fallback returned {len(top_results)} latest reviews")
    return top_results

# ── STEP 4: FORMAT FOR AGENT 2 PROMPT ────────────────────────────

def format_rag_examples(retrieved_reviews: list[dict]) -> str:
    """
    Formats retrieved reviews into a clean block for Agent 2's prompt.
    Shows rating + text so Agent 2 sees both the voice and the rating pattern.
    Behaviour injection — rating label on its own line makes the
    rating+text relationship explicit to Agent 2.
    """
    if not retrieved_reviews:
        return "No similar past reviews found for this user."

    lines = []
    for i, review in enumerate(retrieved_reviews):
        category_note = " [same category]" if review.get('same_category') else ""
        source_note   = " [fallback — latest reviews]" if review.get('source') == 'fallback_latest' else ""

        lines.append(
            f"Past review {i+1}{category_note}{source_note}\n"
            f"  Rating : {int(review['rating'])}★\n"
            f"  Text   : \"{review['text'].strip()}\""
        )

    return "\n\n".join(lines)


In [44]:
# ── SETUP AND TEST ────────────────────────────────────────────────

# load embedding model once — reuse everywhere
print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("✅ Embedding model loaded")

# build index once — reuse across all evaluations
if os.path.exists(RAG_INDEX_PATH):
    print("Loading existing RAG index...")
    rag_index, rag_metadata = load_rag_index()
else:
    print("Building RAG index from persona_df...")
    rag_index, rag_metadata = build_rag_index(persona_df, embedder)

# ── QUICK RETRIEVAL TEST ──────────────────────────────────────────

test_user_id = persona_df['user_id'].iloc[0]

retrieved = retrieve_for_agent2(
    user_id          = "AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA", #test_user_id,
    item_title       = "Electric Shaver for Women ",
    item_description = " ",
    item_category    = "All Beauty",
    index            = rag_index,
    metadata         = rag_metadata,
    embedding_model  = embedder,
    top_k            = 4
)

print(f"\nRetrieved {len(retrieved)} reviews for test user")
for r in retrieved:
    same = "✅ same category" if r['same_category'] else ""
    print(f"  {r['rating']}★ | sim: {r['similarity_score']:.3f} | {same}")
    print(f"  {r['text'][:100]}...")
    print()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
Building RAG index from persona_df...
Building RAG index...
  Reviews to index: 2,206
  Encoding reviews...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

  Index built: 2,206 vectors, dimension 384
  Saved to /kaggle/working/rag_index.faiss
  ℹ️  No same-category results for AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA in 'All Beauty' — returning best semantic matches

Retrieved 4 reviews for test user
  5.0★ | sim: 0.211 | 
  The negative reviews for this game are kind of an exaggeration so let me try to address the main com...

  4.0★ | sim: 0.172 | 
  A bit overpriced for my blood but overall not bad. Probably one of the most detailed link in some ki...

  4.0★ | sim: 0.141 | 
  It's ok but it's also pretty basic. It almost looks toylike to the point were you think you can just...

  4.0★ | sim: 0.133 | 
  The audio quality of the microphone and headset quite frankly isn't very good but it works for your ...



In [45]:
def inspect_rag_index(metadata: list, n: int = 5):
    """
    Inspect the first n records in the RAG metadata.
    Shows exactly what was stored during index build.
    """
    print(f"Total records in metadata: {len(metadata)}")
    print(f"\nFirst {n} records:")
    print("=" * 60)
    
    for i, record in enumerate(metadata[:n]):
        print(f"\nRecord {i+1}:")
        for key, value in record.items():
            display = repr(value)[:80] if isinstance(value, str) else value
            print(f"  {key:<15} : {display}")
        print(f"  text word count : {len(record['text'].split())}")
    
    print("\n" + "=" * 60)
    print("TEXT POPULATION CHECK:")
    empty_text  = sum(1 for m in metadata if not m['text'].strip())
    filled_text = sum(1 for m in metadata if m['text'].strip())
    print(f"  Records with text    : {filled_text}")
    print(f"  Records without text : {empty_text}")

inspect_rag_index(rag_metadata)

Total records in metadata: 2206

First 5 records:

Record 1:
  user_id         : 'AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA'
  text            : "The negative reviews for this game are kind of an exaggeration so let me try to
  rating          : 5.0
  category        : 'Video Games and Software'
  asin            : 'B01GY35HKE'
  timestamp       : 1490504760.0
  text word count : 442

Record 2:
  user_id         : 'AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA'
  text            : 'While not the best rpg, its important to support small devs like this. This edi
  rating          : 5.0
  category        : 'Video Games and Software'
  asin            : 'B01N9KXGKK'
  timestamp       : 1490605252.0
  text word count : 81

Record 3:
  user_id         : 'AE2A5TMJ6YE6ZNWUAFTC6P5XAHXA'
  text            : "What's not to love about it? All kingdom hearts games in one disk! All in GLORI
  rating          : 5.0
  category        : 'Video Games and Software'
  asin            : 'B06XDZGHWK'
  timestamp       : 1492828666.0
  t

## Evaluation function 
- Evaluation using Review Text Quality (ROUGE /BERTScore) & Rating Accuracy (RMSE)

In [46]:


# ── SET HUGGINGFACE TOKEN ────────────────────────────────────────
HF_TOKEN = HF_Key
os.environ["HF_TOKEN"] = HF_TOKEN


# ── ROUGE ────────────────────────────────────────────────────────

def compute_rouge(generated: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer = True
    )
    scores = scorer.score(reference, generated)
    return {
        'rouge1' : round(scores['rouge1'].fmeasure, 4),
        'rouge2' : round(scores['rouge2'].fmeasure, 4),
        'rougeL' : round(scores['rougeL'].fmeasure, 4),
    }


# ── BERTSCORE ────────────────────────────────────────────────────

def compute_bertscore(generated_list: list, reference_list: list) -> list:
    P, R, F1 = bert_score(
        generated_list,
        reference_list,
        lang       = "en",
        model_type = "roberta-large",
        verbose    = False
    )
    return [round(f.item(), 4) for f in F1]


# ── RMSE ─────────────────────────────────────────────────────────

def compute_rmse(true_ratings: list, pred_ratings: list) -> float:
    true = np.array(true_ratings, dtype=float)
    pred = np.array(pred_ratings, dtype=float)
    return round(float(np.sqrt(np.mean((true - pred) ** 2))), 4)


# ── RATE LIMIT HANDLER ────────────────────────────────────────────

def call_with_backoff(fn, *args, max_retries: int = 5, base_delay: float = 30.0, **kwargs):
    """
    Calls fn(*args, **kwargs) with exponential backoff on rate limit errors.
    Catches 429 Resource Exhausted errors from Gemini API.

    Parameters:
        fn          : the function to call
        max_retries : number of retry attempts before giving up
        base_delay  : starting wait time in seconds (doubles each retry)
    """
    for attempt in range(max_retries + 1):
        try:
            return fn(*args, **kwargs)

        except Exception as e:
            error_str = str(e).lower()

            # detect rate limit errors
            is_rate_limit = any(phrase in error_str for phrase in [
                '429', 'resource exhausted', 'rate limit',
                'quota exceeded', 'too many requests'
            ])

            if is_rate_limit and attempt < max_retries:
                # exponential backoff with jitter
                wait = base_delay * (2 ** attempt) + random.uniform(0, 5)
                print(f"\n  ⏳ Rate limit hit — waiting {wait:.0f}s before retry "
                      f"(attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            # not a rate limit error or out of retries — re-raise
            raise


# ── EVALUATION PIPELINE ──────────────────────────────────────────

def evaluate_pipeline(
    persona_df        : pd.DataFrame,
    val_df            : pd.DataFrame,
    flash_model,
    pro_model,
    rag_index,
    rag_metadata      : list,
    embedding_model,
    nigerian_language : str   = None,
    n_users           : int   = 20,
    verbose           : bool  = True,
    mode              : str   = 'validation',
    pause_between_users: float = 10.0,   # ← seconds to wait between users
    pause_every_n      : int   = 5,      # ← pause longer every n users
    long_pause         : float = 30.0,   # ← longer pause every n users
):
    """
    Evaluation pipeline with rate limit protection.

    Rate limit parameters:
        pause_between_users : seconds to sleep after each user (default 10s)
        pause_every_n       : take a longer break every n users (default 5)
        long_pause          : duration of longer break in seconds (default 30s)

    Gemini Flash free tier  : 15 RPM  → pause_between_users=5
    Gemini Flash paid tier  : 1000 RPM → pause_between_users=2
    Gemini Pro free tier    : 2 RPM   → pause_between_users=30
    Gemini Pro paid tier    : 360 RPM → pause_between_users=5
    """

    if mode == 'test':
        print("⚠️  WARNING: You are evaluating on the TEST SET.")
        print("   Only do this once — at final submission.")
        print("   If you are still tuning, use mode='validation'\n")

    results = []
    skipped = 0

    print("=" * 60)
    print(f"EVALUATION PIPELINE  [{mode.upper()}]")
    print("=" * 60)
    print(f"  Users to evaluate   : {n_users}")
    print(f"  Language mode       : {nigerian_language or 'english'}")
    print(f"  Agent 1             : Gemini Flash (persona)")
    print(f"  Agent 2             : Gemini Pro + RAG (generation)")
    print(f"  Pause between users : {pause_between_users}s")
    print(f"  Long pause every    : {pause_every_n} users ({long_pause}s)")
    print()

    # ── OVERLAP CHECK ─────────────────────────────────────────────
    persona_users = set(persona_df['user_id'].unique())
    val_users     = set(val_df['user_id'].unique())
    eval_users    = list(persona_users & val_users)[:n_users]

    if len(eval_users) == 0:
        print("❌ No overlapping users between persona_df and val_df")
        print(f"   Persona users : {len(persona_users)}")
        print(f"   Val users     : {len(val_users)}")
        return pd.DataFrame(), {}

    print(f"  Eligible users      : {len(eval_users)}")
    est_minutes = (len(eval_users) * pause_between_users) / 60
    print(f"  Estimated runtime   : ~{est_minutes:.1f} minutes (pauses only)")
    print()

    # ── PER USER LOOP ─────────────────────────────────────────────
    for idx, user_id in enumerate(eval_users):

        # ── PERSONA HISTORY ───────────────────────────────────────
        history = persona_df[
            persona_df['user_id'] == user_id
        ].sort_values('timestamp')

        texts   = history['text'].tolist()
        ratings = history['rating'].tolist()

        if len(texts) < 3:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"skipped — insufficient history")
            skipped += 1
            continue

        # ── GROUND TRUTH ──────────────────────────────────────────
        holdout_rows = val_df[val_df['user_id'] == user_id]
        if len(holdout_rows) == 0:
            skipped += 1
            continue

        holdout = holdout_rows.iloc[0]

        def safe_get(row, col, default=''):
            val = row[col] if col in row.index else default
            return default if pd.isna(val) else val

        item_asin        = safe_get(holdout, 'parent_asin', 'UNKNOWN')
        item_title       = safe_get(holdout, 'product_title', f'Product {item_asin}')
        item_description = safe_get(holdout, 'description_text', '')
        item_category    = safe_get(holdout, 'main_category', '')
        true_review      = str(safe_get(holdout, 'text', ''))
        true_rating_raw  = safe_get(holdout, 'rating', 3)

        if not item_description:
            item_description = f"Amazon product: {item_title}"

        try:
            true_rating = int(float(true_rating_raw))
        except (ValueError, TypeError):
            true_rating = 3

        # skip video reviews — artificially low ROUGE
        if '[[VIDEOID' in true_review[:50]:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"skipped — video review ground truth")
            skipped += 1
            continue

        if not true_review or len(true_review.split()) < 5:
            skipped += 1
            continue

        # ── TWO-AGENT SIMULATION WITH BACKOFF ─────────────────────
        try:
            dossier, simulated = call_with_backoff(
                simulate_review_two_agent,
                user_id           = user_id,
                texts             = texts,
                ratings           = ratings,
                item_asin         = item_asin,
                item_title        = item_title,
                item_description  = item_description,
                item_category     = item_category,
                flash_model       = flash_model,
                pro_model         = pro_model,
                rag_index         = rag_index,
                rag_metadata      = rag_metadata,
                embedding_model   = embedding_model,
                nigerian_language = nigerian_language,
                verbose           = False,
                base_delay        = pause_between_users * 2
            )

            generated_review = simulated.review
            pred_rating      = simulated.rating
            reasoning        = simulated.reasoning

        except Exception as e:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"⚠️  failed: {e}")
            skipped += 1
            # still pause after failure — API is struggling
            time.sleep(pause_between_users)
            continue

        if not generated_review or len(generated_review.strip()) < 10:
            skipped += 1
            continue

        # ── ROUGE ─────────────────────────────────────────────────
        rouge_scores = compute_rouge(
            generated = generated_review,
            reference = true_review
        )

        # ── TRAIT CONSISTENCY ─────────────────────────────────────
        trait_hits = 0
        if dossier.deep_traits:
            gen_lower = generated_review.lower()
            for trait in dossier.deep_traits:
                trait_words = [
                    w for w in trait.lower().split()
                    if len(w) > 4
                ]
                if any(w in gen_lower for w in trait_words):
                    trait_hits += 1
            trait_consistency = round(trait_hits / len(dossier.deep_traits), 3)
        else:
            trait_consistency = 0.0

        results.append({
            'user_id'          : user_id,
            'true_rating'      : true_rating,
            'pred_rating'      : pred_rating,
            'rating_error'     : abs(true_rating - pred_rating),
            'true_review'      : true_review,
            'generated_review' : generated_review,
            'reasoning'        : reasoning,
            'rouge1'           : rouge_scores['rouge1'],
            'rouge2'           : rouge_scores['rouge2'],
            'rougeL'           : rouge_scores['rougeL'],
            'bertscore'        : None,
            'trait_consistency': trait_consistency,
            'n_traits_found'   : len(dossier.deep_traits),
            'persona_type'     : dossier.rating_behaviour.tendency.value,
            'writing_style'    : dossier.writing_voice.style.value,
            'simulation_brief' : dossier.simulation_brief,
        })

        if verbose:
            print(
                f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                f"| True: {true_rating}★  Pred: {pred_rating}★ "
                f"| ROUGE-1: {rouge_scores['rouge1']:.3f} "
                f"| Traits hit: {trait_hits}/{len(dossier.deep_traits)}"
            )

        # ── PAUSE BETWEEN USERS ───────────────────────────────────
        is_last_user = (idx + 1 == len(eval_users))

        if not is_last_user:
            # longer pause every n users
            if (idx + 1) % pause_every_n == 0:
                print(f"\n  ⏸  Long pause after {idx+1} users — "
                      f"waiting {long_pause}s to reset rate limit window...")
                time.sleep(long_pause)
            else:
                time.sleep(pause_between_users)

    # ── BERTSCORE IN BATCH ────────────────────────────────────────
    if results:
        print(f"\nRunning BERTScore on {len(results)} pairs...")
        try:
            bert_scores = compute_bertscore(
                generated_list = [r['generated_review'] for r in results],
                reference_list = [r['true_review']      for r in results]
            )
            for i, score in enumerate(bert_scores):
                results[i]['bertscore'] = score
        except Exception as e:
            print(f"⚠️  BERTScore failed: {e}")

    # ── BUILD RESULTS DATAFRAME ───────────────────────────────────
    results_df = pd.DataFrame(results)

    if len(results_df) == 0:
        print("❌ No results generated")
        return results_df, {}

    bert_mean = results_df['bertscore'].mean()
    if pd.isna(bert_mean):
        print("⚠️  All BERTScores are None — check roberta-large load")
        results_df['bertscore'] = 0.0
    else:
        results_df['bertscore'] = results_df['bertscore'].fillna(bert_mean)

    # ── SUMMARY ───────────────────────────────────────────────────
    summary = {
        'mode'                 : mode,
        'n_evaluated'          : len(results_df),
        'n_skipped'            : skipped,
        'rmse'                 : compute_rmse(
                                     results_df['true_rating'].tolist(),
                                     results_df['pred_rating'].tolist()
                                 ),
        'mae'                  : round(results_df['rating_error'].mean(), 4),
        'avg_rouge1'           : round(results_df['rouge1'].mean(),           4),
        'avg_rouge2'           : round(results_df['rouge2'].mean(),           4),
        'avg_rougeL'           : round(results_df['rougeL'].mean(),           4),
        'avg_bertscore'        : round(results_df['bertscore'].mean(),        4),
        'avg_trait_consistency': round(results_df['trait_consistency'].mean(), 4),
    }

    print(f"\n{'=' * 60}")
    print(f"RESULTS  [{mode.upper()}]")
    print(f"{'=' * 60}")
    print(f"  Users evaluated      : {summary['n_evaluated']}")
    print(f"  Users skipped        : {summary['n_skipped']}")
    print(f"\n  ── RATING ──────────────────────────────")
    print(f"  RMSE                 : {summary['rmse']}")
    print(f"  MAE                  : {summary['mae']}")
    print(f"  (target RMSE < 1.0  |  strong < 0.8)")
    print(f"\n  ── TEXT QUALITY ────────────────────────")
    print(f"  Avg ROUGE-1          : {summary['avg_rouge1']}")
    print(f"  Avg ROUGE-2          : {summary['avg_rouge2']}")
    print(f"  Avg ROUGE-L          : {summary['avg_rougeL']}")
    print(f"  Avg BERTScore        : {summary['avg_bertscore']}")
    print(f"  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)")
    print(f"\n  ── BEHAVIOURAL FIDELITY ────────────────")
    print(f"  Avg trait consistency: {summary['avg_trait_consistency']}")
    print(f"  (proxy — % of persona traits reflected in generated review)")
    print(f"{'=' * 60}\n")

    print("TOP 3 — highest BERTScore:")
    best = results_df.nlargest(3, 'bertscore')
    for _, row in best.iterrows():
        print(f"  {row['user_id'][:16]} | BERTScore: {row['bertscore']} "
              f"| ROUGE-1: {row['rouge1']}")

    print("\nBOTTOM 3 — lowest BERTScore (investigate these):")
    worst = results_df.nsmallest(3, 'bertscore')
    for _, row in worst.iterrows():
        print(f"\n  User     : {row['user_id']}")
        print(f"  True     : {row['true_review'][:150]}...")
        print(f"  Generated: {row['generated_review'][:150]}...")
        print(f"  Brief    : {row['simulation_brief'][:120]}...")
        print(f"  BERTScore: {row['bertscore']}  ROUGE-1: {row['rouge1']}")

    return results_df, summary


# ── RUN ───────────────────────────────────────────────────────────
# tune pause settings based on your API tier:
#
# Free tier  (Flash 15 RPM,  Pro 2 RPM)  → pause_between_users=35, long_pause=60
# Paid tier  (Flash 1000 RPM, Pro 360 RPM) → pause_between_users=5,  long_pause=15

results_df, summary = evaluate_pipeline(
    persona_df         = persona_df,
    val_df             = val_df,
    flash_model        = FLASH_MODEL,
    pro_model          = PRO_MODEL,
    rag_index          = rag_index,
    rag_metadata       = rag_metadata,
    embedding_model    = embedder,
    nigerian_language  = None,
    n_users            = 20,
    verbose            = True,
    mode               = 'validation',
    pause_between_users= 35.0,   # ← adjust based on your tier
    pause_every_n      = 5,
    long_pause         = 60.0
)

results_df.to_csv('/kaggle/working/validation_results.csv', index=False)
print(f"Results saved to /kaggle/working/validation_results.csv")


# ── FINAL TEST EVAL — run this ONCE at submission ────────────────
# uncomment only when done tuning

# dossier, review = simulate_review_two_agent(
#     user_id           = test_user_id,
#     texts             = test_reviews['text'].tolist(),
#     ratings           = test_reviews['rating'].tolist(),
#     item_asin         = test_reviews.iloc[-1]['parent_asin'],
#     item_title        = test_reviews.iloc[-1].get('product_title', 'Beauty Product'),
#     item_description  = test_reviews.iloc[-1].get('description_text', ''),
#     item_category     = test_reviews.iloc[-1].get('main_category', ''),
#     flash_model       = flash_model,
#     pro_model         = pro_model,
#     rag_index         = rag_index,
#     rag_metadata      = rag_metadata,
#     embedding_model   = embedder,
#     verbose           = True
# )
# test_results_df.to_csv('/kaggle/working/test_results_final.csv', index=False)

EVALUATION PIPELINE  [VALIDATION]
  Users to evaluate   : 20
  Language mode       : english
  Agent 1             : Gemini Flash (persona)
  Agent 2             : Gemini Pro + RAG (generation)
  Pause between users : 35.0s
  Long pause every    : 5 users (60.0s)

  Eligible users      : 20
  Estimated runtime   : ~11.7 minutes (pauses only)

  ⚠️  ItemCard enrichment failed (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}) — using metadata fallback
[Agent 2] Generating with RAG + Pro...
[1/20] AF2WNSSZUVLPCM... | Tru

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS  [VALIDATION]
  Users evaluated      : 20
  Users skipped        : 0

  ── RATING ──────────────────────────────
  RMSE                 : 1.0
  MAE                  : 0.7
  (target RMSE < 1.0  |  strong < 0.8)

  ── TEXT QUALITY ────────────────────────
  Avg ROUGE-1          : 0.3024
  Avg ROUGE-2          : 0.0482
  Avg ROUGE-L          : 0.146
  Avg BERTScore        : 0.8406
  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)

  ── BEHAVIOURAL FIDELITY ────────────────
  Avg trait consistency: 0.55
  (proxy — % of persona traits reflected in generated review)

TOP 3 — highest BERTScore:
  AGUTZC4GHLTGYHA3 | BERTScore: 0.8594 | ROUGE-1: 0.2938
  AEHIZBU7DVRLXGTW | BERTScore: 0.8573 | ROUGE-1: 0.2902
  AHY7NSZXW4IUPQ2E | BERTScore: 0.8556 | ROUGE-1: 0.2727

BOTTOM 3 — lowest BERTScore (investigate these):

  User     : AFSJ74DFC2CDZPTZTSDSSYGINVNA
  True     : So, first off I'll say i love visual novels and went in expecting to read a lot.  Which is true, but my issue is

## swap api method for evaluation pipeline 

In [47]:
from google import genai

In [48]:
import time
import random
from typing import Optional
import pandas as pd
from google import genai  # Modern google-genai SDK

# ── API KEY POOL ──────────────────────────────────────────────────

class GeminiKeyManager:
    """
    Manages a pool of Gemini API keys using the modern google-genai Client.
    Automatically rotates to the next key when a rate limit is hit.
    Tracks cooldown per key so exhausted keys are not retried too soon.
    """

    def __init__(self, api_keys: list[str], cooldown_seconds: float = 62.0):
        """
        Parameters:
            api_keys         : list of Gemini API keys
            cooldown_seconds : how long to wait before retrying an exhausted key
        """
        if not api_keys:
            raise ValueError("At least one API key required")

        self.api_keys         = api_keys
        self.cooldown_seconds = cooldown_seconds
        self.current_index    = 0
        self.exhausted_at     = {}   # key → timestamp when it was exhausted

        # Initialize the first client instance directly
        self.client = genai.Client(api_key=self.get_current_key())
        print(f"GeminiKeyManager initialised with {len(api_keys)} key(s)")

    def _is_cooling_down(self, key: str) -> bool:
        """Check if a key is still in cooldown."""
        if key not in self.exhausted_at:
            return False
        elapsed = time.time() - self.exhausted_at[key]
        return elapsed < self.cooldown_seconds

    def _time_remaining(self, key: str) -> float:
        """Seconds remaining in cooldown for a key."""
        if key not in self.exhausted_at:
            return 0.0
        elapsed  = time.time() - self.exhausted_at[key]
        remaining = self.cooldown_seconds - elapsed
        return max(0.0, remaining)

    def get_current_key(self) -> str:
        """Returns the current active API key string."""
        return self.api_keys[self.current_index]

    def mark_exhausted(self, key: str):
        """Mark a key as rate-limited and rotate to the next available key."""
        self.exhausted_at[key] = time.time()
        print(f"\n  🔄 Key ...{key[-6:]} rate limited — rotating...")
        self._rotate()

    def _rotate(self):
        """Find the next available key that is not in cooldown and update the client."""
        original_index = self.current_index
        attempts       = 0

        while attempts < len(self.api_keys):
            self.current_index = (self.current_index + 1) % len(self.api_keys)
            candidate          = self.api_keys[self.current_index]

            if not self._is_cooling_down(candidate):
                print(f"  ✅ Switched to key ...{candidate[-6:]}")
                # Update our managed client with the new active key
                self.client = genai.Client(api_key=candidate)
                return

            remaining = self._time_remaining(candidate)
            print(f"  ⏳ Key ...{candidate[-6:]} still cooling down "
                  f"({remaining:.0f}s remaining)")
            attempts += 1

        # All keys exhausted — wait for the one that recovers soonest
        soonest_key = min(
            self.api_keys,
            key=lambda k: self._time_remaining(k)
        )
        wait = self._time_remaining(soonest_key) + 2   # small buffer
        print(f"\n  ⚠️  All keys exhausted — waiting {wait:.0f}s for "
              f"key ...{soonest_key[-6:]} to recover...")
        time.sleep(wait)
        
        self.current_index = self.api_keys.index(soonest_key)
        self.client = genai.Client(api_key=soonest_key)
        print(f"  ✅ Resumed with key ...{soonest_key[-6:]}")

    def status(self):
        """Print current status of all keys."""
        print("\nAPI KEY STATUS:")
        for i, key in enumerate(self.api_keys):
            marker = "← active" if i == self.current_index else ""
            if self._is_cooling_down(key):
                remaining = self._time_remaining(key)
                print(f"  Key ...{key[-6:]} | cooling down ({remaining:.0f}s) {marker}")
            else:
                print(f"  Key ...{key[-6:]} | ready {marker}")


# ── SMART API CALL WITH KEY ROTATION ─────────────────────────────

def call_with_key_rotation(
    fn,
    key_manager : GeminiKeyManager,
    *args,
    max_retries : int   = 10,
    min_pause   : float = 2.0,
    **kwargs
):
    """
    Calls fn(*args, **kwargs) and rotates API key on rate limit.
    Retries until success or max_retries exhausted.
    """
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)

        except Exception as e:
            error_str = str(e).lower()

            is_rate_limit = any(phrase in error_str for phrase in [
                '429', 'resource exhausted', 'rate limit',
                'quota exceeded', 'too many requests',
                'ratequotaexceeded'
            ])

            if is_rate_limit:
                current_key = key_manager.get_current_key()
                key_manager.mark_exhausted(current_key)

                # small pause before retry with new key
                pause = min_pause + random.uniform(0, 2)
                time.sleep(pause)
                continue

            # not a rate limit error — re-raise immediately
            raise

    raise RuntimeError(f"All {max_retries} retry attempts failed")


# ── UPDATED EVALUATION PIPELINE CALL SITE ────────────────────────

def evaluate_pipeline(
    persona_df        : pd.DataFrame,
    val_df            : pd.DataFrame,
    flash_model_name  : str,
    pro_model_name    : str,
    rag_index,
    rag_metadata      : list,
    embedding_model,
    key_manager       : GeminiKeyManager,
    nigerian_language : str  = None,
    n_users           : int  = 20,
    verbose           : bool = True,
    mode              : str  = 'validation',
):
    if mode == 'test':
        print("⚠️  WARNING: You are evaluating on the TEST SET.")
        print("   Only do this once — at final submission.")
        print("   If you are still tuning, use mode='validation'\n")

    results = []
    skipped = 0

    print("=" * 60)
    print(f"EVALUATION PIPELINE  [{mode.upper()}]")
    print("=" * 60)
    print(f"  Users to evaluate : {n_users}")
    print(f"  Language mode     : {nigerian_language or 'english'}")
    print(f"  Agent 1           : {flash_model_name} (persona)")
    print(f"  Agent 2           : {pro_model_name} + RAG (generation)")
    print(f"  API keys in pool  : {len(key_manager.api_keys)}")
    print()

    # ── OVERLAP CHECK ─────────────────────────────────────────────
    persona_users = set(persona_df['user_id'].unique())
    val_users     = set(val_df['user_id'].unique())
    eval_users    = list(persona_users & val_users)[:n_users]

    if len(eval_users) == 0:
        print("❌ No overlapping users between persona_df and val_df")
        print(f"   Persona users : {len(persona_users)}")
        print(f"   Val users     : {len(val_users)}")
        return pd.DataFrame(), {}

    print(f"  Eligible users    : {len(eval_users)}")
    print()

    for idx, user_id in enumerate(eval_users):

        history = persona_df[
            persona_df['user_id'] == user_id
        ].sort_values('timestamp')

        texts   = history['text'].tolist()
        ratings = history['rating'].tolist()

        if len(texts) < 3:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"skipped — insufficient history")
            skipped += 1
            continue

        holdout_rows = val_df[val_df['user_id'] == user_id]
        if len(holdout_rows) == 0:
            skipped += 1
            continue

        holdout = holdout_rows.iloc[0]

        def safe_get(row, col, default=''):
            val = row[col] if col in row.index else default
            return default if pd.isna(val) else val

        item_asin        = safe_get(holdout, 'parent_asin', 'UNKNOWN')
        item_title       = safe_get(holdout, 'product_title', f'Product {item_asin}')
        item_description = safe_get(holdout, 'description_text', '')
        item_category    = safe_get(holdout, 'main_category', '')
        true_review      = str(safe_get(holdout, 'text', ''))
        true_rating_raw  = safe_get(holdout, 'rating', 3)

        if not item_description:
            item_description = f"Amazon product: {item_title}"

        try:
            true_rating = int(float(true_rating_raw))
        except (ValueError, TypeError):
            true_rating = 3

        # skip video reviews
        if '[[VIDEOID' in true_review[:50]:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"skipped — video review")
            skipped += 1
            continue

        if not true_review or len(true_review.split()) < 5:
            skipped += 1
            continue

        # ── CALL WITH KEY ROTATION ────────────────────────────────
        try:
            # key_manager positional parameter placement fixed here:
            dossier, simulated = call_with_key_rotation(
                simulate_review_two_agent,
                key_manager,              
                user_id           = user_id,
                texts             = texts,
                ratings           = ratings,
                item_asin         = item_asin,
                item_title        = item_title,
                item_description  = item_description,
                item_category     = item_category,
                flash_model       = flash_model_name,  
                pro_model         = pro_model_name,    
                rag_index         = rag_index,
                rag_metadata      = rag_metadata,
                embedding_model   = embedding_model,
                nigerian_language = nigerian_language,
                verbose           = False
            )

            generated_review = simulated.review
            pred_rating      = simulated.rating
            reasoning        = simulated.reasoning

        except Exception as e:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                      f"⚠️  failed: {e}")
            skipped += 1
            continue

        if not generated_review or len(generated_review.strip()) < 10:
            skipped += 1
            continue

        rouge_scores = compute_rouge(
            generated = generated_review,
            reference = true_review
        )

        trait_hits = 0
        if dossier.deep_traits:
            gen_lower = generated_review.lower()
            for trait in dossier.deep_traits:
                trait_words = [
                    w for w in trait.lower().split()
                    if len(w) > 4
                ]
                if any(w in gen_lower for w in trait_words):
                    trait_hits += 1
            trait_consistency = round(trait_hits / len(dossier.deep_traits), 3)
        else:
            trait_consistency = 0.0

        results.append({
            'user_id'          : user_id,
            'true_rating'      : true_rating,
            'pred_rating'      : pred_rating,
            'rating_error'     : abs(true_rating - pred_rating),
            'true_review'      : true_review,
            'generated_review' : generated_review,
            'reasoning'        : reasoning,
            'rouge1'           : rouge_scores['rouge1'],
            'rouge2'           : rouge_scores['rouge2'],
            'rougeL'           : rouge_scores['rougeL'],
            'bertscore'        : None,
            'trait_consistency': trait_consistency,
            'n_traits_found'   : len(dossier.deep_traits),
            'persona_type'     : dossier.rating_behaviour.tendency.value,
            'writing_style'    : dossier.writing_voice.style.value,
            'simulation_brief' : dossier.simulation_brief,
            'api_key_used'     : f"...{key_manager.get_current_key()[-6:]}"
        })

        if verbose:
            print(
                f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                f"| True: {true_rating}★  Pred: {pred_rating}★ "
                f"| ROUGE-1: {rouge_scores['rouge1']:.3f} "
                f"| Traits hit: {trait_hits}/{len(dossier.deep_traits)} "
                f"| Key: ...{key_manager.get_current_key()[-6:]}"
            )

    # ── BERTSCORE IN BATCH ────────────────────────────────────────
    if results:
        print(f"\nRunning BERTScore on {len(results)} pairs...")
        try:
            bert_scores = compute_bertscore(
                generated_list = [r['generated_review'] for r in results],
                reference_list = [r['true_review']      for r in results]
            )
            for i, score in enumerate(bert_scores):
                results[i]['bertscore'] = score
        except Exception as e:
            print(f"⚠️  BERTScore failed: {e}")

    results_df = pd.DataFrame(results)

    if len(results_df) == 0:
        print("❌ No results generated")
        return results_df, {}

    bert_mean = results_df['bertscore'].mean()
    if pd.isna(bert_mean):
        print("⚠️  All BERTScores are None")
        results_df['bertscore'] = 0.0
    else:
        results_df['bertscore'] = results_df['bertscore'].fillna(bert_mean)

    summary = {
        'mode'                 : mode,
        'n_evaluated'          : len(results_df),
        'n_skipped'            : skipped,
        'rmse'                 : compute_rmse(
                                     results_df['true_rating'].tolist(),
                                     results_df['pred_rating'].tolist()
                                 ),
        'mae'                  : round(results_df['rating_error'].mean(), 4),
        'avg_rouge1'           : round(results_df['rouge1'].mean(),           4),
        'avg_rouge2'           : round(results_df['rouge2'].mean(),           4),
        'avg_rougeL'           : round(results_df['rougeL'].mean(),           4),
        'avg_bertscore'        : round(results_df['bertscore'].mean(),        4),
        'avg_trait_consistency': round(results_df['trait_consistency'].mean(), 4),
    }

    print(f"\n{'=' * 60}")
    print(f"RESULTS  [{mode.upper()}]")
    print(f"{'=' * 60}")
    print(f"  Users evaluated      : {summary['n_evaluated']}")
    print(f"  Users skipped        : {summary['n_skipped']}")
    print(f"\n  ── RATING ──────────────────────────────")
    print(f"  RMSE                 : {summary['rmse']}")
    print(f"  MAE                  : {summary['mae']}")
    print(f"  (target RMSE < 1.0  |  strong < 0.8)")
    print(f"\n  ── TEXT QUALITY ────────────────────────")
    print(f"  Avg ROUGE-1          : {summary['avg_rouge1']}")
    print(f"  Avg ROUGE-2          : {summary['avg_rouge2']}")
    print(f"  Avg ROUGE-L          : {summary['avg_rougeL']}")
    print(f"  Avg BERTScore        : {summary['avg_bertscore']}")
    print(f"  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)")
    print(f"\n  ── BEHAVIOURAL FIDELITY ────────────────")
    print(f"  Avg trait consistency: {summary['avg_trait_consistency']}")
    print(f"{'=' * 60}\n")

    if 'api_key_used' in results_df.columns:
        print("API KEY USAGE:")
        print(results_df['api_key_used'].value_counts().to_string())

    print("\nTOP 3 — highest BERTScore:")
    best = results_df.nlargest(3, 'bertscore')
    for _, row in best.iterrows():
        print(f"  {row['user_id'][:16]} | BERTScore: {row['bertscore']} "
              f"| ROUGE-1: {row['rouge1']}")

    print("\nBOTTOM 3 — lowest BERTScore:")
    worst = results_df.nsmallest(3, 'bertscore')
    for _, row in worst.iterrows():
        print(f"\n  User     : {row['user_id']}")
        print(f"  True     : {row['true_review'][:150]}...")
        print(f"  Generated: {row['generated_review'][:150]}...")
        print(f"  Brief    : {row['simulation_brief'][:120]}...")
        print(f"  BERTScore: {row['bertscore']}  ROUGE-1: {row['rouge1']}")

    return results_df, summary


In [49]:
# ── RUN ───────────────────────────────────────────────────────────

# 1. Initialize the Key Manager (It builds the initial genai.Client internally)
key_manager = GeminiKeyManager(
    api_keys = [
        "YOUR_PRIMARY_KEY",
        "YOUR_BACKUP_KEY_1",
        "YOUR_BACKUP_KEY_2",
    ],
    cooldown_seconds = 62.0
)

# 2. Assign the string model target IDs as variables
FLASH_MODEL_NAME = "gemini-2.0-flash"
PRO_MODEL_NAME   = "gemini-2.0-flash" 

# 3. Fire the pipeline execution
results_df, summary = evaluate_pipeline(
    persona_df        = persona_df,
    val_df            = val_df,
    flash_model_name  = FLASH_MODEL_NAME,  # <-- FIX: Match the updated function definition parameter name
    pro_model_name    = PRO_MODEL_NAME,    # <-- FIX: Match the updated function definition parameter name
    rag_index         = rag_index,
    rag_metadata      = rag_metadata,
    embedding_model   = embedder,
    key_manager       = key_manager,
    nigerian_language = None,
    n_users           = 20,
    verbose           = True,
    mode              = 'validation'
)

GeminiKeyManager initialised with 3 key(s)
EVALUATION PIPELINE  [VALIDATION]
  Users to evaluate : 20
  Language mode     : english
  Agent 1           : gemini-2.0-flash (persona)
  Agent 2           : gemini-2.0-flash + RAG (generation)
  API keys in pool  : 3

  Eligible users    : 20

[Agent 2] Generating with RAG + Pro...
[1/20] AF2WNSSZUVLPCM... | True: 5★  Pred: 5★ | ROUGE-1: 0.297 | Traits hit: 4/4 | Key: ...RY_KEY
[Agent 2] Generating with RAG + Pro...
[2/20] AE2A5TMJ6YE6ZN... | True: 5★  Pred: 4★ | ROUGE-1: 0.320 | Traits hit: 1/4 | Key: ...RY_KEY
[Agent 2] Generating with RAG + Pro...
[3/20] AHH3BR4LDVBEAV... | True: 5★  Pred: 5★ | ROUGE-1: 0.276 | Traits hit: 2/4 | Key: ...RY_KEY
[Agent 2] Generating with RAG + Pro...
[4/20] AFUB4MRXTUAADQ... | True: 3★  Pred: 4★ | ROUGE-1: 0.178 | Traits hit: 1/4 | Key: ...RY_KEY
[Agent 2] Generating with RAG + Pro...
[5/20] AGN77Y4XMITXHD... | True: 5★  Pred: 4★ | ROUGE-1: 0.185 | Traits hit: 2/4 | Key: ...RY_KEY
[Agent 2] Generating with

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS  [VALIDATION]
  Users evaluated      : 20
  Users skipped        : 0

  ── RATING ──────────────────────────────
  RMSE                 : 1.0247
  MAE                  : 0.75
  (target RMSE < 1.0  |  strong < 0.8)

  ── TEXT QUALITY ────────────────────────
  Avg ROUGE-1          : 0.291
  Avg ROUGE-2          : 0.0402
  Avg ROUGE-L          : 0.1439
  Avg BERTScore        : 0.8392
  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)

  ── BEHAVIOURAL FIDELITY ────────────────
  Avg trait consistency: 0.575

API KEY USAGE:
api_key_used
...RY_KEY    20

TOP 3 — highest BERTScore:
  AGUTZC4GHLTGYHA3 | BERTScore: 0.8573 | ROUGE-1: 0.2599
  AEHIZBU7DVRLXGTW | BERTScore: 0.8492 | ROUGE-1: 0.2626
  AFUB4MRXTUAADQOJ | BERTScore: 0.848 | ROUGE-1: 0.1778

BOTTOM 3 — lowest BERTScore:

  User     : AGWB4F74ER6FISJQKYS2FG6BVSTQ
  True     : It's kind of funny, I don't have that much gray in my beard but there's some and no one has ever said anything about it looking bad. The few that

## nigerian contexualization evaluation

In [50]:
def evaluate_nigerian_comparison(
    persona_df       : pd.DataFrame,
    val_df           : pd.DataFrame,
    flash_model_name : str,
    pro_model_name   : str,
    rag_index,
    rag_metadata     : list,
    embedding_model,
    key_manager      : GeminiKeyManager,  # ← Added to support Client Key Rotation
    n_users          : int  = 20,
    verbose          : bool = False
) -> dict:
    """
    Runs evaluation twice — once without Nigerian conditioning,
    once with — and produces a side-by-side comparison.

    This directly supports the solution paper's Nigerian contextualisation
    claim by showing it as a measurable architectural decision.
    """

    print("=" * 60)
    print("NIGERIAN MODE COMPARISON EVALUATION")
    print("=" * 60)
    print(f"Users per mode : {n_users}")
    print(f"Modes          : standard english vs nigerian conditioning")
    print()

    # ── RUN STANDARD ─────────────────────────────────────────────
    print("▶ Running STANDARD mode...")
    results_standard, summary_standard = evaluate_pipeline(
        persona_df        = persona_df,
        val_df            = val_df,
        flash_model_name  = flash_model_name,
        pro_model_name    = pro_model_name,
        rag_index         = rag_index,
        rag_metadata      = rag_metadata,
        embedding_model   = embedding_model,
        key_manager       = key_manager,      # ← Pass down to maintain Client instance
        nigerian_language = None,
        n_users           = n_users,
        verbose           = verbose,
        mode              = 'validation'
    )

    # ── RUN NIGERIAN ──────────────────────────────────────────────
    print("\n▶ Running NIGERIAN mode...")
    results_nigerian, summary_nigerian = evaluate_pipeline(
        persona_df        = persona_df,
        val_df            = val_df,
        flash_model_name  = flash_model_name,
        pro_model_name    = pro_model_name,
        rag_index         = rag_index,
        rag_metadata      = rag_metadata,
        embedding_model   = embedding_model,
        key_manager       = key_manager,      # ← Pass down to maintain Client instance
        nigerian_language = 'pidgin',
        n_users           = n_users,
        verbose           = verbose,
        mode              = 'validation'
    )

    # ── GUARD: ensure both runs have results ──────────────────────
    if len(results_standard) == 0 or len(results_nigerian) == 0:
        print("❌ One or both runs returned no results — check pipeline")
        return {}

    # ── COMPUTE DELTAS ────────────────────────────────────────────
    metrics = ['rmse', 'mae', 'avg_rouge1', 'avg_rouge2',
               'avg_rougeL', 'avg_bertscore', 'avg_trait_consistency']

    deltas = {}
    for metric in metrics:
        std_val = summary_standard.get(metric, 0)
        nig_val = summary_nigerian.get(metric, 0)
        if metric in ('rmse', 'mae'):
            deltas[metric] = round(std_val - nig_val, 4)
        else:
            deltas[metric] = round(nig_val - std_val, 4)

    # ── PRINT COMPARISON ──────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"NIGERIAN MODE COMPARISON RESULTS")
    print(f"{'=' * 60}")
    print(f"{'Metric':<28} {'Standard':>10} {'Nigerian':>10} {'Delta':>10}")
    print(f"{'─' * 60}")

    metric_labels = {
        'rmse'                 : 'RMSE (↓ better)',
        'mae'                  : 'MAE  (↓ better)',
        'avg_rouge1'           : 'ROUGE-1 (↑ better)',
        'avg_rouge2'           : 'ROUGE-2 (↑ better)',
        'avg_rougeL'           : 'ROUGE-L (↑ better)',
        'avg_bertscore'        : 'BERTScore (↑ better)',
        'avg_trait_consistency': 'Trait consistency (↑)',
    }

    for metric, label in metric_labels.items():
        std_val = summary_standard.get(metric, 0)
        nig_val = summary_nigerian.get(metric, 0)
        delta   = deltas[metric]
        arrow   = "✅ +" if delta > 0 else ("➖  " if delta == 0 else "⚠️  ")
        print(f"  {label:<26} {std_val:>10.4f} {nig_val:>10.4f} {arrow}{abs(delta):.4f}")

    print(f"<b>─</b>" * 60)

    # ── QUALITATIVE COMPARISON ────────────────────────────────────
    print(f"\nSAMPLE REVIEW COMPARISON (first user):")
    print(f"{'─' * 60}")

    if len(results_standard) > 0 and len(results_nigerian) > 0:
        std_row = results_standard.iloc[0]
        nig_row = results_nigerian[
            results_nigerian['user_id'] == std_row['user_id']
        ]

        print(f"User     : {std_row['user_id']}")
        print(f"True     : {std_row['true_review'][:200]}...")
        print(f"\nStandard : {std_row['generated_review'][:200]}...")

        if len(nig_row) > 0:
            print(f"\nNigerian : {nig_row.iloc[0]['generated_review'][:200]}...")
        print()

    # ── INTERPRETATION ────────────────────────────────────────────
    print("INTERPRETATION:")
    improvements = sum(1 for d in deltas.values() if d > 0)
    regressions  = sum(1 for d in deltas.values() if d < 0)

    if improvements >= 5:
        print("  ✅ Nigerian conditioning improves most metrics")
        print("     Strong evidence for architectural claim in paper")
    elif improvements >= 3:
        print("  ➖ Nigerian conditioning shows mixed results")
        print("     Some metrics improve — note tradeoffs in paper")
    else:
        print("  ⚠️  Nigerian conditioning hurts most metrics")
        print("     May indicate prompt needs tuning for this dataset")
        print("     Consider: Amazon US data may not reflect Nigerian patterns")

    print(f"\n  Metrics improved : {improvements}/{len(metrics)}")
    print(f"  Metrics hurt     : {regressions}/{len(metrics)}")

    # ── SAVE BOTH RESULTS ─────────────────────────────────────────
    results_standard.to_csv('/kaggle/working/results_standard.csv', index=False)
    results_nigerian.to_csv('/kaggle/working/results_nigerian.csv', index=False)
    print(f"\nResults saved:")
    print(f"  /kaggle/working/results_standard.csv")
    print(f"  /kaggle/working/results_nigerian.csv")

    return {
        'summary_standard'  : summary_standard,
        'summary_nigerian'  : summary_nigerian,
        'deltas'            : deltas,
        'results_standard'  : results_standard,
        'results_nigerian'  : results_nigerian,
    }


# ── RUN THE COMPARISON ────────────────────────────────────────────

comparison = evaluate_nigerian_comparison(
    persona_df       = persona_df,
    val_df           = val_df,
    flash_model_name = FLASH_MODEL_NAME, # ← Fixed argument name to match your string variables
    pro_model_name   = PRO_MODEL_NAME,   # ← Fixed argument name to match your string variables
    rag_index        = rag_index,
    rag_metadata     = rag_metadata,
    embedding_model  = embedder,
    key_manager      = key_manager,      # ← Injected key manager interface
    n_users          = 20,
    verbose          = False
)

NIGERIAN MODE COMPARISON EVALUATION
Users per mode : 20
Modes          : standard english vs nigerian conditioning

▶ Running STANDARD mode...
EVALUATION PIPELINE  [VALIDATION]
  Users to evaluate : 20
  Language mode     : english
  Agent 1           : gemini-2.0-flash (persona)
  Agent 2           : gemini-2.0-flash + RAG (generation)
  API keys in pool  : 3

  Eligible users    : 20

[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with RAG + Pro...
[Agent 2] Generating with

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS  [VALIDATION]
  Users evaluated      : 20
  Users skipped        : 0

  ── RATING ──────────────────────────────
  RMSE                 : 1.0
  MAE                  : 0.7
  (target RMSE < 1.0  |  strong < 0.8)

  ── TEXT QUALITY ────────────────────────
  Avg ROUGE-1          : 0.3036
  Avg ROUGE-2          : 0.0457
  Avg ROUGE-L          : 0.1461
  Avg BERTScore        : 0.8411
  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)

  ── BEHAVIOURAL FIDELITY ────────────────
  Avg trait consistency: 0.5625

API KEY USAGE:
api_key_used
...RY_KEY    20

TOP 3 — highest BERTScore:
  AGUTZC4GHLTGYHA3 | BERTScore: 0.8691 | ROUGE-1: 0.4048
  AHY7NSZXW4IUPQ2E | BERTScore: 0.8567 | ROUGE-1: 0.349
  AEHIZBU7DVRLXGTW | BERTScore: 0.8564 | ROUGE-1: 0.2887

BOTTOM 3 — lowest BERTScore:

  User     : AGWB4F74ER6FISJQKYS2FG6BVSTQ
  True     : It's kind of funny, I don't have that much gray in my beard but there's some and no one has ever said anything about it looking bad. The few that c

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS  [VALIDATION]
  Users evaluated      : 20
  Users skipped        : 0

  ── RATING ──────────────────────────────
  RMSE                 : 0.9747
  MAE                  : 0.65
  (target RMSE < 1.0  |  strong < 0.8)

  ── TEXT QUALITY ────────────────────────
  Avg ROUGE-1          : 0.2925
  Avg ROUGE-2          : 0.038
  Avg ROUGE-L          : 0.1421
  Avg BERTScore        : 0.8357
  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)

  ── BEHAVIOURAL FIDELITY ────────────────
  Avg trait consistency: 0.55

API KEY USAGE:
api_key_used
...RY_KEY    20

TOP 3 — highest BERTScore:
  AEHIZBU7DVRLXGTW | BERTScore: 0.8498 | ROUGE-1: 0.2626
  AGUTZC4GHLTGYHA3 | BERTScore: 0.8485 | ROUGE-1: 0.321
  AFUNTTQHT7DCP4AA | BERTScore: 0.8481 | ROUGE-1: 0.299

BOTTOM 3 — lowest BERTScore:

  User     : AEHOFUNZP6VT74RUDDCJ2VVIT56A
  True     : For the 'where I'm coming from' on this review, I've been playing a lot of Demon's Souls and Dark Souls 1 and 2 in the past few years. I love the s

In [51]:
# ── SAVE PIPELINE ARTIFACTS TO KAGGLE WORKING DIRECTORY ──
import os
import pickle
import faiss

print("Exporting production RAG assets to disk...")

# 1. Define your export destination paths matching your pipeline configuration
EXPORT_INDEX_PATH = '/kaggle/working/rag_index.faiss'
EXPORT_META_PATH  = '/kaggle/working/rag_meta.pkl'
EXPORT_CACHE_DIR  = '/kaggle/working/persona_cache/'

# 2. Save the FAISS vector index structure directly using its internal writer
try:
    faiss.write_index(rag_index, EXPORT_INDEX_PATH)
    print(f"    Saved FAISS index successfully -> {EXPORT_INDEX_PATH} ({rag_index.ntotal:,} vectors)")
except NameError:
    print("     Error: 'rag_index' variable not found in memory. Ensure the index block ran successfully.")

# 3. Save the companion text/ID metadata matching the vectors using pickle
try:
    with open(EXPORT_META_PATH, 'wb') as f:
        pickle.dump(rag_metadata, f)
    print(f"    Saved RAG metadata dictionary map -> {EXPORT_META_PATH} ({len(rag_metadata):,} items)")
except NameError:
    print("    Error: 'rag_metadata' variable not found in memory.")

# 4. Verify your pre-compiled user persona profiles cache folder structure exists
if os.path.exists(EXPORT_CACHE_DIR):
    cached_profiles_count = len(os.listdir(EXPORT_CACHE_DIR))
    print(f"    Found persona profiles folder -> {EXPORT_CACHE_DIR} ({cached_profiles_count} JSON profiles cached)")
else:
    os.makedirs(EXPORT_CACHE_DIR, exist_ok=True)
    print(f"     Created fresh persona profile directory template -> {EXPORT_CACHE_DIR}")

print("\n All Task A pipeline structures are safely preserved. You can now download them directly from the Kaggle Output side panel!")

Exporting production RAG assets to disk...
    Saved FAISS index successfully -> /kaggle/working/rag_index.faiss (2,206 vectors)
    Saved RAG metadata dictionary map -> /kaggle/working/rag_meta.pkl (2,206 items)
    Found persona profiles folder -> /kaggle/working/persona_cache/ (20 JSON profiles cached)

 All Task A pipeline structures are safely preserved. You can now download them directly from the Kaggle Output side panel!


In [52]:
import os
import shutil
import faiss
import pickle

# 1. Archive the persona cache directory
CACHE_DIR = '/kaggle/working/persona_cache/'
ZIP_OUTPUT = '/kaggle/working/persona_cache_archive'
if os.path.exists(CACHE_DIR) and len(os.listdir(CACHE_DIR)) > 0:
    shutil.make_archive(ZIP_OUTPUT, 'zip', CACHE_DIR)
    print("✅ Created persona_cache_archive.zip")

# 2. Force write the index and metadata to /kaggle/working/
faiss.write_index(rag_index, '/kaggle/working/rag_index.faiss')
with open('/kaggle/working/rag_meta.pkl', 'wb') as f:
    pickle.dump(rag_metadata, f)

print("💾 All assets are successfully staged on Kaggle's virtual disk. Ready for Colab pull!")

✅ Created persona_cache_archive.zip
💾 All assets are successfully staged on Kaggle's virtual disk. Ready for Colab pull!
